# 🚀 Olist Logistics Intelligence - Executive Report

> **Complete ML pipeline: Inventory optimization + Distribution network analysis**

---

## ⚙️ Quick Start

### Option 1: Google Colab ⭐ (Recommended)
```python
# 1. Upload notebook to Colab
# 2. Run first cell (mounts Drive + configures paths)
# 3. Download dataset from Kaggle (if not already in Drive)
# 4. Run all cells sequentially
```

**First run:** ~20 min (trains models from scratch)  
**Subsequent runs:** ~5 min (uses cached models)

### Option 2: Local Jupyter
```bash
# 1. Clone repo
git clone https://github.com/magooscar86/olist-logistics-intelligence
cd olist-logistics-intelligence

# 2. Install dependencies
pip install -r requirements.txt

# 3. Download dataset
# https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
# Place CSVs in: data/raw/

# 4. Launch Jupyter
jupyter lab

# 5. Open and run: notebooks/Olist_Executive_Report.ipynb
```

---

## 📊 What This Notebook Does

### Phase 1-2: Data & EDA
- Loads 110k orders from Olist marketplace
- Feature engineering & temporal analysis

### Phase 3: Forecasting Tournament
- Compares GP, LightGBM, Moving Average
- Selects best model per category (5-fold CV)

### Phase 4-5: Inventory Optimization
- Calculates optimal safety stock
- Dynamic inventory by service level

### Phase 6: Spatial Intelligence
- Trains Kriging model (RBF kernel)
- Predicts delays at ANY coordinate

### Phase 7: Facility Location
- K-Means weighted by "logistical pain"
- Proposes optimal hub locations

### Phase 8: Network Analytics
- Graph analysis (4k nodes, 5k edges)
- Identifies vulnerabilities

---

## 💾 Cache Behavior

The notebook is **smart about caching**:

**If cache exists** (`checkpoints/` folder):
- Loads pre-trained models instantly
- Skips expensive computations

**If no cache** (first run or GitHub clone):
- Trains everything from scratch
- Saves results for future runs
- Takes ~20 minutes

**Cache files** (automatically created):
- `checkpoints/df_main.parquet` (~17 MB)
- `checkpoints/gp_spatial_model.pkl` (~120 MB)
- `checkpoints/facility_results.pkl` (~30 KB)

---

## ⚠️ Known Limitations

- Spatial model training is memory-intensive (needs 4GB RAM)
- First run downloads ~160 MB of CSV data
- Some visualizations require internet (Folium maps)

---

## 🆘 Troubleshooting

**Error: "ModuleNotFoundError: No module named 'src'"**
→ Run the configuration cell (first code cell)

**Error: "FileNotFoundError: data/raw/olist_orders_dataset.csv"**
→ Download Olist dataset from Kaggle

**Notebook runs slow:**
→ Normal for first run. Use Colab GPU (Runtime → Change runtime type → GPU)

---


# 🚀 OLIST LOGISTICS INTELLIGENCE
## Sistema Completo de Forecasting e Inteligencia Espacial

**Proyecto:** Machine Learning para Optimización de Cadena de Suministro  
**Dataset:** Olist E-Commerce (Brasil, 2016-2018)  
**Autor:** [Oscar Antonio Melo Leon]  
**Fecha:** Enero 2026

---

## 📋 EXECUTIVE SUMMARY

Sistema end-to-end de inteligencia logística que combina:

✅ **Forecasting Temporal (Fase 2)**
- 5 modelos comparados rigurosamente
- Cross-validation + pruebas estadísticas
- Selección automática por categoría
- Optimización de inventario
- Mejora promedio: **15.5%** vs baselines

✅ **Inteligencia Espacial (Fase 3)**
- Kriging espacial (Gaussian Processes)
- Interpolación de riesgo logístico
- Radio de influencia: **~111 km**
- Detección de zonas críticas
- Mapas interactivos con incertidumbre

**Impacto de Negocio:**
- Reducción de **~20%** en stock innecesario
- Predicción de retrasos en coordenadas sin historial
- Sistema de alertas por zona geográfica

In [ ]:
# ============================================================================
# CONFIGURACIÓN - Ejecutar esta celda primero
# ============================================================================

from pathlib import Path
import sys
import os

# Detectar entorno (Colab vs Local)
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    # Montar Google Drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Olist_Project')
    print("🌐 Modo: Google Colab")
else:
    # Ruta local relativa - detecta automáticamente la raíz del proyecto
    # Asumiendo que se ejecuta desde la raíz del proyecto o desde notebooks/
    PROJECT_ROOT = Path.cwd()
    # Si estamos en notebooks/, subir un nivel
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    print("💻 Modo: Jupyter Local")
    print(f"   Ruta detectada: {PROJECT_ROOT}")

# Añadir a path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root: {PROJECT_ROOT}")

# Verificar existencia de datos
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
CHECKPOINTS = PROJECT_ROOT / 'checkpoints'

print(f"📊 Verificando datos:")
print(f"   Data raw: {'✅' if DATA_RAW.exists() else '❌'} {DATA_RAW}")
print(f"   Checkpoints: {'✅' if CHECKPOINTS.exists() else '⚠️ (se crearán)'} {CHECKPOINTS}")

# Crear directorios si no existen
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'outputs').mkdir(exist_ok=True)

print(f"💡 Modo de ejecución:")
if (CHECKPOINTS / 'df_main.parquet').exists():
    print("   🚀 RÁPIDO - Usará cache existente (~5 min)")
else:
    print("   🐢 COMPLETO - Entrenará desde cero (~20 min)")


---
# 📦 PARTE 1: CONFIGURACIÓN
---

In [ ]:
# ============================================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================================

import sys
from pathlib import Path

# Detectar si estamos en Google Colab
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    # Montar Google Drive solo en Colab
    print("📁 Montando Google Drive...")
    drive.mount('/content/drive', force_remount=False)
    project_root = Path(str(PROJECT_ROOT))
    print("🌐 Modo: Google Colab")
else:
    # Ruta local - ajustar según tu estructura
    project_root = Path.cwd()
    print("💻 Modo: Jupyter Local")

if not project_root.exists():
    raise FileNotFoundError(f"❌ Proyecto no encontrado en: {project_root}")

# Agregar al sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Entorno configurado")
print(f"   Project Root: {project_root}")
print(f"   Python buscará módulos en: {sys.path[0]}")


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Image
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Librerías cargadas")

---
# 📊 PARTE 2: CARGA DE DATOS
---

In [ ]:
# Carga de datos con fallback automático
try:
    # Intentar carga rápida desde cache
    from notebooks.quick_load import quick_load
    print("⚡ Cargando desde cache...")
    data = quick_load(load_spatial=True, load_forecasting=False)
    df_main = data['df_main']
    print(f"   ✅ df_main cargado: {df_main.shape}")

except (FileNotFoundError, ImportError, KeyError) as e:
    print(f"⚠️ Cache no disponible: {e}")
    print("🔄 Cargando y procesando datos desde cero...")

    # Cargar desde CSV raw
    from src.data.data_loader import OlistDataLoader
    from src.data.feature_engineering import FeatureEngineer
    
    # Cargar todos los CSVs
    loader = OlistDataLoader(raw_data_path=str(PROJECT_ROOT / 'data' / 'raw'))
    data = loader.load_all()
    
    # Fusionar datasets
    engineer = FeatureEngineer()
    df_main = engineer.process(
        orders=data['orders'],
        customers=data['customers'],
        geo=data['geo'],
        items=data['items'],
        products=data['products']
    )

    # Guardar para futuros usos
    checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'df_main.parquet'
    df_main.to_parquet(checkpoint_path)
    print(f"   ✅ Guardado en: {checkpoint_path}")


---
# 🏆 PARTE 3: TORNEO DE MODELOS (FASE 2)
---

## Metodología

Comparamos **5 modelos** en **6 categorías** principales:

| Modelo | Tipo | Validación |
|--------|------|------------|
| Media Móvil | Baseline | CV (5 folds) |
| Holt-Winters | Estacional | CV (5 folds) |
| LightGBM | ML | CV (5 folds) |
| GP Puro | Bayesiano | Hold-out (75/12) |
| Híbrido (GP+LGBM) | Ensemble | Hold-out (75/12) |

**Razón del split:** GP requiere ≥1 ciclo estacional (52 semanas) para entrenar correctamente.

In [ ]:
# Cargar resultados
df_results = pd.read_csv(str(PROJECT_ROOT / 'outputs/cv_results_final.csv'))
df_summary = pd.read_csv(str(PROJECT_ROOT / 'outputs/model_selection_summary.csv'))

print(f"✅ Resultados: {len(df_results)} evaluaciones")
print(f"✅ Categorías: {df_summary['categoria'].nunique()}")

---
# 🏆 PARTE 3: TORNEO DE MODELOS (FASE 2)
---

## 🎯 Objetivo y Contexto

Este torneo evalúa **5 algoritmos de forecasting** en **6 categorías de productos** para determinar cuál es el mejor modelo para cada categoría específica.

### ¿Por qué un "torneo"?

En forecasting temporal, **no existe un modelo universal superior**. La volatilidad, estacionalidad y tendencia varían por categoría, por lo que necesitamos un **sistema selector inteligente** que elija el mejor modelo por caso de uso.

---

## 🔬 Metodología

### Modelos Evaluados

Comparamos **5 enfoques** que representan diferentes filosofías de predicción:

| Modelo | Tipo | Filosofía | Cuándo funciona mejor |
|--------|------|-----------|----------------------|
| **Media Móvil** | Baseline estadístico | Promedio simple de ventana deslizante | Series estables sin tendencia |
| **Holt-Winters** | Suavizamiento exponencial | Captura tendencia + estacionalidad | Patrones cíclicos claros |
| **LightGBM Puro** | Gradient Boosting (ML) | Aprende patrones no lineales | Volatilidad alta, cambios abruptos |
| **GP Puro** | Proceso Gaussiano (Bayesiano) | Modelado probabilístico con incertidumbre | Tendencias suaves, necesita contexto temporal largo |
| **Híbrido (GP+LGBM)** | Ensemble | Combina suavidad (GP) + cortes (LGBM) | Casos complejos con múltiples patrones |

---

### Estrategia de Validación

**¿Por qué dos métodos diferentes?**
```python
# MODELOS RÁPIDOS (MA, HW, LGBM)
método = "Cross-Validation (5 folds)"
razón = "Se entrenan en segundos → podemos hacer 5 particiones"

# MODELOS COSTOSOS (GP Puro, Híbrido)
método = "Hold-Out (75/25)"
razón = "GP requiere ≥52 semanas de entrenamiento → 1 sola partición"
```

**Rigor científico:**
- ✅ **Validación temporal** (no aleatoria): El modelo predice el futuro, no el pasado mezclado
- ✅ **Walk-forward:** Cada fold simula un escenario de producción real
- ✅ **Anti-leakage:** Ninguna información futura contamina el entrenamiento

<details>
<summary>🔍 ¿Qué es Cross-Validation temporal? (click para expandir)</summary>

**Problema con CV tradicional:**
En series temporales, NO podemos entrenar en 2020 y validar en 2018. Rompe la causalidad.

**Solución: Time Series Split**
```
Fold 1: ████████ train | ▓▓ test
Fold 2: ██████████ train | ▓▓ test
Fold 3: ████████████ train | ▓▓ test
Fold 4: ██████████████ train | ▓▓ test
Fold 5: ████████████████ train | ▓▓ test
```

El modelo "avanza en el tiempo" probando su capacidad de generalizar al futuro.

**Resultado:**
- RMSE_mean = Error promedio en los 5 tests
- RMSE_std = Variabilidad (qué tan consistente es)

</details>

---

### Métrica: RMSE (Root Mean Squared Error)

**¿Qué mide?**
Error promedio de predicción en **unidades del producto**.

**Ejemplo concreto:**
- RMSE = 26.3 unidades (Housewares)
- Si predigo vender 100 → Real estará entre **74-126** (±26)

**¿Por qué RMSE y no MAE?**
- ✅ Penaliza errores grandes más fuerte (crítico en inventario)
- ✅ Diferenciable (mejor para optimización)
- ✅ Estándar industrial (comparable con literatura)

---

## 📊 Dashboard de Resultados

## Dashboard de Resultados

In [ ]:
# ============================================================================
# DASHBOARD DE TORNEO - VERSIÓN 3.7 (RIGOR CIENTÍFICO)
# ============================================================================

# --- SETUP ---
import sys
from pathlib import Path

project_root = Path(str(PROJECT_ROOT))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# --- IMPORTS ---
from src.visualization.tournament_viz import TournamentVisualizer
import pandas as pd

print("🎨 Generando Dashboard Ejecutivo v3.7...")

# --- CARGAR DATOS ---
df_results = pd.read_csv(project_root / 'outputs' / 'cv_results_final.csv')

# --- GENERAR DASHBOARD ---
viz = TournamentVisualizer()

viz.create_winners_dashboard(
    results_df=df_results,
    save_path=str(project_root / 'outputs' / 'dashboard_torneo_v3.png')
)

print("\n✅ Dashboard actualizado")
print("📁 Guardado en: outputs/dashboard_torneo_v3.png")

---

## 🔍 Interpretación del Dashboard v3.7

### Panel A: Ranking de Precisión

**Insights clave:**

✅ **Housewares: RMSE = 26.3** (Campeón absoluto)
- Categoría más predecible del dataset
- Demanda estable, poca volatilidad
- Ganador: **LightGBM Puro** (CV)

⚠️ **Computers: RMSE = 46.0** (Mayor dificultad)
- Alta volatilidad de demanda
- Sorprendentemente ganó **Media Móvil** (baseline)
- Señal: Comportamiento casi aleatorio (ruido blanco)

🎯 **Patrón observado:**
Categorías de uso diario (Housewares, Health) son más predecibles que tecnología o moda.

---

### Panel B: Valor Agregado (Mejora vs Baseline)

**Hallazgos críticos:**

🏆 **Furniture: +32.4% de mejora**
- GP Puro superó dramáticamente al baseline
- Razón: Tendencia suave capturada por kernel gaussiano
- **Impacto de negocio:** 32% menos inventario desperdiciado

✅ **Bed & Bath: +26.3%** | **Housewares: +17.6%** | **Health: +16.7%**
- Mejoras significativas con ML
- Justifica inversión en modelos complejos

❌ **Computers: Baseline** | **Sports: Baseline**
- ML **NO mejoró** sobre Media Móvil
- **Conclusión:** A veces simplicidad > complejidad
- Estas categorías tienen alta componente aleatoria

**Insight estratégico:**
> No implementamos ML "porque sí". En 2/6 categorías, un promedio simple funciona mejor.
> Esto demuestra **rigor científico** sobre marketing de IA.

---

### Panel C: Mapa de Consistencia

**¿Qué estamos viendo?**
- **Eje X:** Error promedio (menor = mejor)
- **Eje Y:** Inestabilidad entre folds (menor = mejor)
- **Zona verde:** Alto rendimiento (bajo error + alta consistencia)

**⚠️ NOTA METODOLÓGICA:**
Solo aparecen modelos validados con **Cross-Validation**.

GP Puro y Híbrido (Hold-Out) se excluyen porque NO tienen desviación estándar calculada en múltiples particiones. Esto es **rigor científico**, no una limitación.

**Análisis:**

✅ **Housewares** (zona verde):
- Bajo RMSE (26.3) + Baja variabilidad (σ ≈ 8)
- Predicción **confiable y estable**
- Ideal para decisiones de inventario críticas

⚠️ **Sports** (zona amarilla):
- RMSE moderado (36.6) pero alta variabilidad (σ ≈ 11.5)
- El modelo es **menos confiable**
- Recomendación: Aumentar stock de seguridad

⚠️ **Computers** (zona alta):
- Alta inestabilidad (σ ≈ 12)
- Predicciones inconsistentes entre folds
- Confirma: Categoría difícil de predecir

**Interpretación de negocio:**
> Housewares puede operar con inventario Just-In-Time (bajo riesgo).
> Sports requiere buffer alto (alta incertidumbre).

---

### Panel D: Distribución de Estrategias

**Resultados del torneo:**

📊 **Empate triple:**
- 🟢 **LightGBM Puro:** 33% (2 categorías)
- 🔵 **GP Puro:** 33% (2 categorías)
- ⚪ **Media Móvil:** 33% (2 categorías)

❌ **Fracaso total:**
- 🟣 **Híbrido (GP+LGBM):** 0% (0 categorías)

**¿Por qué falló el Ensemble?**

El híbrido intentó combinar:
- Suavidad del GP (para tendencias)
- Cortes del LGBM (para cambios abruptos)

**Problema:**
En forecasting, combinar modelos puede **promediar la señal** en lugar de capturar lo mejor de ambos.

**Lección aprendida:**
> Ensemble NO siempre mejora. En nuestro caso, especializarse > generalizar.

---

## 📈 Análisis de Ganadores por Categoría

### Victoria del GP Puro (Furniture, Bed & Bath)

**Características comunes:**
- ✅ Tendencias suaves y graduales
- ✅ Estacionalidad predecible
- ✅ Pocas disrupciones

**Por qué ganó GP:**
El kernel RBF (Radial Basis Function) del Gaussian Process modela correlaciones temporales de largo alcance. Capturó ciclos anuales mejor que árboles de decisión.

**Código conceptual:**
```python
# GP aprende: "Ventas en t+52 semanas correlacionan con t"
kernel = RBF(length_scale=52) + WhiteKernel(noise_level=σ²)
```

---

### Victoria de LightGBM (Health, Housewares)

**Características comunes:**
- ⚡ Cambios de demanda abruptos
- 📊 Patrones no lineales
- 🎯 Features adicionales importantes (precio, promociones)

**Por qué ganó LGBM:**
Los árboles de decisión gradient-boosted pueden hacer "cortes" bruscos en el espacio de features. Perfectos para detectar:
- Efectos de promociones
- Cambios de temporada repentinos
- Interacciones complejas entre variables

---

### Victoria de Media Móvil (Computers, Sports)

**🚨 HALLAZGO CRÍTICO:**

Estas categorías se comportan casi como **ruido blanco** en horizontes cortos.

**Interpretación:**
- Alta volatilidad intrínseca
- Componente aleatoria dominante
- Cualquier modelo complejo **overfittea** el ruido

**Conclusión:**
> En presencia de alta incertidumbre, la simplicidad del promedio móvil **generaliza mejor**.
> Esto valida nuestra metodología: probamos todo y dejamos que los datos decidan.

---

## 🎯 Conclusiones del Torneo

### 1. No existe un modelo universal

La distribución **33%-33%-33%** confirma: necesitamos un **sistema selector** que elija por categoría.

### 2. Simplicidad a veces gana

Media Móvil ganó 2/6 categorías. No siempre "más complejo" = mejor.

### 3. Validación cruzada es crucial

CV reveló inestabilidades ocultas que Hold-Out no habría detectado.

### 4. Ensemble no es magia

Híbrido fracasó en todas las categorías. Combinar modelos requiere más que sumarlos.

---

## 💰 Impacto de Negocio

**Ahorro estimado en inventario:**
- Furniture: 32.4% menos stock innecesario
- Promedio general: **~20% reducción** en costos de inventario

**Reducción de stockouts:**
- Mejora en predicción → Mejor planificación
- Estimado: **~15% menos quiebres de stock**

**ROI del proyecto:**
- Inversión: Tiempo de desarrollo + infraestructura
- Retorno: 20% ahorro en $M de inventario anual
- Payback: < 6 meses

---

In [ ]:
Image(str(PROJECT_ROOT / 'outputs/heatmap_comparacion.png'))

## Análisis de Ganadores

In [ ]:
# Tabla de ganadores
winners = df_results.loc[df_results.groupby("Categoría")["RMSE_mean"].idxmin()]

display(winners[['Categoría', 'Modelo', 'RMSE_mean', 'Método']].style
    .background_gradient(subset=['RMSE_mean'], cmap='RdYlGn_r')
    .format({'RMSE_mean': '{:.2f}'})
)

# Distribución
print("\n📊 Distribución de Victorias:")
print(winners['Modelo'].value_counts())

---
# 📈 PARTE 4: VALIDACIÓN ESTADÍSTICA
---

---
# 📈 PARTE 4: VALIDACIÓN ESTADÍSTICA
---

## 🎯 ¿Por qué Friedman Test?

El torneo mostró que diferentes modelos ganan en diferentes categorías. Pero **¿estas diferencias son reales o casualidad?**

Para responderlo con rigor científico, aplicamos el **Test de Friedman** - el estándar en comparación múltiple de algoritmos ML.

---

## 🔬 Metodología

**Test de Friedman (1937):**
- ✅ No paramétrico (no asume normalidad)
- ✅ Datos pareados (mismas categorías para todos)
- ✅ Comparación múltiple (5 modelos simultáneos)

**Hipótesis:**
- **H₀:** Todos los modelos son equivalentes
- **H₁:** Al menos un modelo es significativamente superior

**Criterio:** Si p-value < 0.05 → Rechazamos H₀

---

## 📊 Resultados

In [ ]:
from src.models.statistical_tests import StatisticalTester

tester = StatisticalTester(df_results, alpha=0.05)

pivot = df_results.pivot(index='Categoría', columns='Modelo', values='RMSE_mean')
friedman = tester.friedman_test(pivot)

print("PRUEBA DE FRIEDMAN (Comparación Global)")
print(f"p-value: {friedman['p_value']:.4f}")
print(f"\n{friedman['interpretation']}")
print("\nRankings:")
display(friedman['rankings'])

---

## 🔍 Interpretación

### Resultado: p = 0.1468 > 0.05

**Conclusión:** NO hay evidencia de que un modelo sea **universalmente superior**.

---

### ¿Es esto bueno o malo?

✅ **¡ES EXCELENTE!** Aquí está el por qué:

**1. Valida el sistema selector**
- Si p < 0.05 → Un modelo siempre gana → NO necesitamos selector
- Como p > 0.05 → Cada modelo tiene su nicho → **Selector justificado** ✅

**2. Confirma teoría ML**
- **No Free Lunch Theorem** (Wolpert, 1997): "No existe algoritmo óptimo para todos los problemas"
- Nuestro resultado empírico **confirma** este teorema

**3. Demuestra rigor científico**
- ❌ Enfoque ingenuo: "LightGBM es mejor, úsalo siempre"
- ✅ Nuestro enfoque: Probamos estadísticamente y descubrimos que cada modelo brilla en diferentes contextos

---

### Rankings Promedio

| Modelo | Ranking | Interpretación |
|--------|---------|----------------|
| LightGBM Puro | 1.83 | Mejor en promedio (no significativo) |
| Media Móvil | 2.50 | Baseline sólido |
| Holt-Winters | 3.33 | Competitivo en estacionalidad |
| GP Puro | 3.33 | Excelente en tendencias suaves |
| Híbrido | 4.00 | Fracaso consistente |

**Insights:**
- ⭐ LGBM lidera pero NO estadísticamente superior
- ⚖️ GP y HW operan en nichos diferentes
- ❌ Ensemble falló (NO siempre mejora)
- 🎯 Baseline ganó 2/6 categorías (simplicidad funciona)

---

## 💼 Impacto de Negocio

**Sistema selector vs modelo único:**
- Con selector: ~20% ahorro en inventario
- Modelo único: ~8% ahorro
- **Diferencia: 12% = Justifica desarrollo completo**

**En presentaciones:**

❌ Débil: "Nuestro modelo es el mejor"

✅ Profesional: "Evaluamos 5 modelos con validación estadística (Friedman, p=0.1468). Descubrimos que NO existe ganador universal, por eso desarrollamos sistema selector logrando 20% ahorro vs enfoque único."

---

## 📚 Referencia Académica

**Estándar ML:** Demšar, J. (2006). "Statistical Comparisons of Classifiers over Multiple Data Sets." *JMLR* (5000+ citas)

**Nuestra implementación:** Sigue metodología recomendada ✅


---
# 💰 PARTE 5: OPTIMIZACIÓN DE INVENTARIO
---

In [ ]:
# ============================================================================
# CONFIGURACIÓN PROFESIONAL - EJECUTAR UNA SOLA VEZ
# ============================================================================

from pathlib import Path
import warnings

project_root = Path(str(PROJECT_ROOT))
viz_dir = project_root / 'src' / 'visualization'

# --- 1. ACTUALIZAR __init__.py ---
init_content = '''"""
Paquete de visualización - Olist Logistics Intelligence
"""

from .forecasting_viz import ForecastingVisualizer
from .tournament_viz import TournamentVisualizer
from .spatial_viz import SpatialVisualizer
from .spatial_analytics import SpatialAnalytics
from .interactive_ui import OlistDashboard
from .inventory_report import InventoryReportGenerator

__all__ = [
    'ForecastingVisualizer',
    'TournamentVisualizer',
    'SpatialVisualizer',
    'SpatialAnalytics',
    'OlistDashboard',
    'InventoryReportGenerator'
]
'''

with open(viz_dir / '__init__.py', 'w', encoding='utf-8') as f:
    f.write(init_content)

print("✅ Configuración completada")

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE INVENTARIO
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configuración limpia
warnings.filterwarnings('ignore')
project_root = Path(str(PROJECT_ROOT))

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar módulo
from src.visualization.inventory_report import InventoryReportGenerator

# Cargar y preparar datos
plans_df = pd.read_csv(project_root / 'outputs' / 'inventory_plans.csv')
standard = plans_df[plans_df['Nivel Servicio'] == '95%'].copy()
standard['% Riesgo'] = (standard['Stock Seguridad'] / standard['Cantidad Pedido'] * 100).round(1)

# Generar reporte (guardar sin mostrar)
print("🎨 Generando reporte ejecutivo...")
generator = InventoryReportGenerator()
generator.create_executive_report(
    df_inventory=standard,
    save_path=str(project_root / 'outputs' / 'reporte_financiero_v2.png'),
    show_plot=False  # No mostrar automáticamente
)

# Cerrar figura de matplotlib (evitar duplicados)
plt.close('all')

# Mostrar imagen guardada
print("✅ Reporte generado\n")
output_path = project_root / 'outputs' / 'reporte_financiero_v2.png'
display(Image(filename=str(output_path), width=1200))

# 📊 PARTE 5: OPTIMIZACIÓN FINANCIERA DE INVENTARIO
## Eficiencia del Capital de Trabajo

### 5.1 Resumen Ejecutivo
La implementación de la **Arquitectura de Selección Inteligente** (asignando dinámicamente entre Media Móvil, GP o LightGBM según la categoría) ha permitido radiografiar el costo financiero del riesgo en la cadena de suministro.

> **Hallazgo Clave:** El costo de proteger el Nivel de Servicio (95%) no es proporcional al volumen de ventas, sino a la **previsibilidad intrínseca** de la categoría.

---

### 5.2 Nota Metodológica: Cuantificación del Riesgo ($\sigma$)

Para traducir las predicciones matemáticas en **Dinero (Stock de Seguridad)**, utilizamos una estrategia híbrida de cuantificación de incertidumbre. Esto asegura que el "colchón" financiero sea coherente con la naturaleza del modelo ganador:

1.  **Escenario Probabilístico (Cuando gana GP):**
    *   Utilizamos la **Desviación Estándar Predictiva ($\sigma_{pred}$)** nativa del modelo Bayesiano. El riesgo es dinámico y crece si intentamos planificar muy a futuro.
2.  **Escenario Determinista (Cuando gana Media Móvil o LightGBM):**
    *   Al ser modelos que entregan un solo número, utilizamos el **RMSE de Validación** como proxy de la volatilidad futura ($\sigma \approx RMSE_{test}$).
    *   *Lógica:* Asumimos la **Estacionariedad del Error**: si el modelo falló por 40 unidades en el pasado reciente, reservamos capital para cubrir ese margen de error.

> **Fórmula de Impacto Financiero:**
> $$ \text{Capital en Riesgo} = \text{Costo Unitario} \times (Z_{95\%} \times \sigma_{\text{modelo}} \times \sqrt{\text{Lead Time}}) $$

---

### 5.3 Hallazgos Estratégicos (Matriz Riesgo-Volumen)

#### A. Categorías "Cash Cows" (Alta Eficiencia)
**Líderes:** `bed_bath_table` (18.3% riesgo) y `housewares` (19.0% riesgo).

*   **Diagnóstico:**
    *   Presentan el menor índice de riesgo del portafolio (**<19%**).
    *   Los modelos lograron capturar sus patrones estructurales con alta precisión (bajo RMSE), permitiendo operar con un inventario "delgado" y eficiente.
*   **Recomendación:**
    *   ✅ **Mantener política actual** de servicio 95%.
    *   💰 Aprovechar la estabilidad de la demanda para negociar compras anticipadas (Economía de Escala).

#### B. Categorías Problemáticas ⚠️ (Atención Prioritaria)
**Crítico:** `computers_accessories` (26.8% de riesgo).

*   **Modelo Ganador:** **Media Móvil**.
*   **Diagnóstico:**
    *   A pesar de tener una demanda media-baja (87 unidades/semana), es la categoría que **más capital inmoviliza en stock de seguridad ($6,400 USD)**.
    *   La victoria de la Media Móvil confirma que su comportamiento es altamente estocástico (ruido blanco), obligando al sistema a inflar el inventario para protegerse.
*   **Acciones Recomendadas:**
    *   🔍 **Auditoría de Cadena:** Investigar si la volatilidad proviene de la demanda o de tiempos de entrega irregulares de proveedores.
    *   💡 **Ajuste Táctico:** Evaluar reducir el Nivel de Servicio al **90%** exclusivamente para esta categoría. Esto liberaría capital inmediato con un impacto marginal en ventas perdidas.

---

### 5.4 Impacto Financiero Total


**Costo Total de Riesgo: $31,300 USD**  
(Capital inmovilizado en stock de seguridad para nivel de servicio 95%)

#### 📊 Comparativa vs Baseline

**Sistema Actual (Media Móvil):**
- Costo de stock de seguridad: $38,000 USD
- Método: Promedio simple sin optimización por categoría

**Sistema Propuesto (Selector Inteligente):**
- Costo de stock de seguridad: **$31,300 USD**
- Método: GP + LightGBM con selección automática por categoría

**💰 Impacto financiero:**
- **Ahorro anual: $6,700 USD**
- **Reducción: 18% en capital inmovilizado**
- **ROI estimado: Payback < 6 meses**

---



### ### 4. Insight Estratégico Clave 🎯

> **"Optimizar solo la categoría `computers_accessories` (reduciendo su volatilidad
> o ajustando su nivel de servicio) liberaría más capital ($1,200 USD) que optimizar
> las tres categorías más eficientes juntas."**

Esto valida la importancia del **sistema selector inteligente de modelos**:

#### 📊 En categorías predecibles (Hogar, Bed & Bath):

- **Modelos avanzados funcionan excepcionalmente bien:**
  - Gaussian Process (GP) captura patrones estacionales suaves
  - LightGBM detecta cambios abruptos en demanda
  - RMSE bajo (26-40 unidades) = **Alta confianza en predicciones**

- **Resultado:** Necesitamos **menos stock de seguridad** (18-19% del pedido)
- **Estrategia:** Mantener nivel de servicio 95% y negociar descuentos por volumen

#### ⚠️ En categorías volátiles (Tecnología, Accesorios):

- **El torneo de modelos reveló algo crítico:**
  - Incluso los modelos más sofisticados (GP, LightGBM) tuvieron RMSE alto (46.0)
  - **La Media Móvil (baseline simple) ganó** - señal de comportamiento casi aleatorio
  - Alta varianza σ² = **Volatilidad intrínseca de la demanda**

- **Diagnóstico:** El problema **no es el algoritmo de predicción**, sino la
  **inestabilidad estructural** de la cadena de suministro

- **Estrategia:** Intervención de negocio necesaria:
  1. Auditar confiabilidad de proveedores (lead times erráticos)
  2. Considerar bajar nivel de servicio de 95% → 90%
  3. Evaluar modelo de reposición frecuente vs pedidos grandes

---

#### 💡 Lección del Sistema ML

El sistema de forecasting inteligente **no solo predice** - también **diagnostica**:

| Lo que el modelo SÍ puede resolver | Lo que requiere acción de negocio |
|-------------------------------------|-----------------------------------|
| ✅ Capturar patrones estacionales | ❌ Proveedores poco confiables |
| ✅ Ajustar por tendencias | ❌ Demanda estructuralmente caótica |
| ✅ Optimizar inventario en categorías estables | ❌ Problemas de calidad/disponibilidad |

**Conclusión:** Un modelo que te dice "no puedo predecir esto con confianza"
es tan valioso como uno que predice perfectamente - **te indica dónde enfocar
recursos humanos** en lugar de mejores algoritmos.


---
# 🎨 PARTE 5.5: INTERFAZ INTERACTIVA DE FORECASTING
---

## Sistema de Comando y Control

Esta interfaz permite:
- **Selección de categoría** (6 categorías principales)
- **Ajuste de horizonte temporal** (adaptativo según memoria GP)
- **Nivel de servicio** (80%, 95%, 99%)
- **Cálculo en tiempo real** de orden sugerida
- **Visualización histórica + proyección**

**Características técnicas:**
- Horizonte máximo ajustado por la "memoria predictiva" del GP
- Stock de seguridad calibrado por incertidumbre del modelo
- Rango probable basado en desviación estándar

In [ ]:
# Preparar datos para interfaz
print("⚙️ Preparando interfaz interactiva...")

# Si df_main no está en memoria, cargar desde checkpoint
if 'df_main' not in locals():
    print("   Cargando df_main...")
    df_main = pd.read_parquet(str(PROJECT_ROOT / 'checkpoints/df_main.parquet'))
    print(f"   ✅ df_main cargado: {df_main.shape}")

print("✅ Datos preparados")

# 5.5 Contexto Industrial: El Valor del Enfoque Probabilístico en "Small Data"

Para validar la relevancia de esta herramienta en un entorno corporativo, debemos contrastar nuestra metodología con los estándares actuales de la industria (ERP tradicionales) y las tendencias de vanguardia (Amazon).

### 1. El Referente de Vanguardia: Probabilistic Forecasting (Amazon DeepAR)
Las empresas líderes en logística, como Amazon, han abandonado los pronósticos de "número único". Utilizan algoritmos como **DeepAR**, que generan **pronósticos probabilísticos**.
*   **El Concepto:** En lugar de predecir que la demanda será exactamente 100 unidades, el sistema predice una distribución de probabilidad (ej. una Campana de Gauss a futuro).
*   **La Ventaja:** Esto permite automatizar decisiones financieras: *"Compra inventario suficiente para cubrir el 95% de la probabilidad, pero no gastes dinero en cubrir el 99% si el producto es barato"*.
*   **Nuestra Implementación:** Nuestro modelo de **Procesos Gaussianos (GP)** replica esta filosofía probabilística de Amazon, pero adaptada matemáticamente para funcionar con **historiales cortos**, donde las redes neuronales masivas no son viables.

### 2. La Ventaja Competitiva: El Problema del "Small Data"
La mayoría de los sistemas legacy (Oracle SCM, SAP APO) utilizan algoritmos paramétricos clásicos como **Holt-Winters** o **Medias Móviles**. Estos sistemas tienen una debilidad crítica: **Requieren mucha historia.**
*   **La Limitación de Oracle/SAP:** Para detectar estacionalidad (ej. Navidad), métodos como Holt-Winters requieren matemáticamente al menos **2 ciclos completos** (24-36 meses) de historia limpia. Si el producto tiene solo 14 meses de vida, el sistema falla o vuelve a un promedio simple ineficiente.
*   **La Solución del Proceso Gaussiano:** Nuestro modelo no "aprende" parámetros fijos; utiliza un **Kernel** (función de covarianza) que define cómo se relacionan los puntos.
    *   **Resultado:** Puede inferir tendencias y bandas de riesgo coherentes con tan solo **30-40 semanas** de datos. Es ideal para productos nuevos, categorías de nicho o mercados emergentes donde la historia es escasa.

### 3. Matriz de Aplicabilidad: ¿Cuándo usar este modelo?

No proponemos reemplazar todo el sistema ERP, sino complementar sus puntos ciegos:

| Escenario | Modelo Sugerido | Justificación |
| :--- | :--- | :--- |
| **Productos Maduros (>3 años historia)** | **Holt-Winters / LGBM** | Hay suficientes datos para que la estadística clásica sea precisa y computacionalmente barata. |
| **Productos Recientes (<1.5 años)** | **Proceso Gaussiano (GP)** | La ventaja del GP en *Small Data* es insuperable. Captura la señal de tendencia sin necesitar ciclos anuales completos. |
| **Demanda Errática/Volátil** | **Proceso Gaussiano** | La Media Móvil es ciega al riesgo. El GP cuantifica la volatilidad ($\sigma$), permitiendo protegerse con Stock de Seguridad. |

### 4. Estrategia de Operación (MLOps)
Dado que el Proceso Gaussiano tiene un costo computacional cúbico $O(N^3)$, su despliegue debe ser táctico:
*   **Frecuencia de Entrenamiento:** **Semanal (Batch).** La estructura del mercado (tendencia/estacionalidad) no cambia diariamente. Entrenar cada noche es un desperdicio de recursos.
*   **Inferencia:** En tiempo real (usando el modelo pre-entrenado la noche anterior).

## 🖥️ Olist AI Command Center

Interfaz interactiva para toma de decisiones estratégicas con optimización de inventario en tiempo real.

<img src="../outputs/Olist%20AI%20Command%20Center.png" width="100%">

**Funcionalidades:**
- 📊 Panel ejecutivo de 4 métricas clave
- 🎯 Optimización de inventario por categoría
- 📈 Predicción de retrasos con modelos Gaussian Process
- ⚙️ Ajuste de nivel de servicio y horizonte de planificación

In [ ]:
# Lanzar interfaz interactiva
from src.visualization.interactive_ui import OlistDashboard

print("🚀 Inicializando dashboard...")
print("⏳ Esto puede tomar 1-2 minutos (entrenando modelos GP)...")

dashboard = OlistDashboard(df_main)
dashboard.render()

print("\n✅ Interfaz lista")
print("\n📌 Instrucciones:")
print("   1. Selecciona una categoría")
print("   2. Ajusta el horizonte de planificación")
print("   3. Elige el nivel de servicio")
print("   4. Click en 'CALCULAR ESTRATEGIA'")

---
# 🎨 PARTE 5.5: SISTEMA DE COMANDO Y CONTROL
## Interfaz Interactiva para Planificación de Inventario
---

## 1. Contexto Industrial: El Valor del Enfoque Probabilístico

Para validar la relevancia de esta herramienta en un entorno corporativo, contrastamos nuestra metodología con los estándares actuales de la industria y las tendencias de vanguardia.

### 🏆 El Referente de Vanguardia: Amazon DeepAR

Las empresas líderes en logística han abandonado los pronósticos de "número único". Utilizan algoritmos como **DeepAR** (Deep Learning), que generan pronósticos probabilísticos.

**El Concepto:**

En lugar de predecir "Venderás 100", el sistema dice: "Hay un 10% de probabilidad de vender 80, un 50% de vender 100, y un 90% de vender menos de 120".

**Nuestra Implementación:**

Nuestro modelo de **Procesos Gaussianos (GP)** replica esta filosofía probabilística, pero adaptada matemáticamente para funcionar con **Small Data** (historiales de <2 años), donde las redes neuronales masivas fallarían.

---

### ⚠️ La Limitación del ERP Tradicional (Oracle/SAP)

La mayoría de los sistemas legacy calculan el stock de seguridad usando desviaciones estándar fijas o buffers manuales. Son "ciegos" a la dinámica del riesgo.

**Nuestra Ventaja:**

El GP calcula la **incertidumbre dinámica**. Sabe que la volatilidad no es constante; crece con el tiempo y varía según la época del año.

---

## 2. Capacidades del Centro de Comando

Esta interfaz operacionaliza los hallazgos matemáticos del proyecto, permitiendo a los planificadores simular estrategias de inventario en tiempo real para cualquiera de las 6 categorías principales.

### ⚙️ Lógica del Algoritmo (Backend)

El sistema ejecuta una arquitectura de decisión en tiempo real:

1. **Selección de Modelo:** Al elegir una categoría, el sistema carga automáticamente el modelo Gaussian Process para calcular la Demanda Base.

2. **Cálculo de Riesgo:** Utiliza el Proceso Gaussiano para calcular el σ (Sigma) y dimensionar el Stock de Seguridad.

3. **Restricción de Horizonte:** El slider de semanas se adapta automáticamente a la "Memoria Predictiva" detectada, impidiendo proyecciones más allá de lo matemáticamente confiable.

---

## 3. Guía de Interpretación de Resultados

Analizamos un caso de uso real extraído de la interfaz para la categoría **Bed Bath & Table**:

### A. El Diagnóstico (Barra Superior)

- **Modelo Seleccionado:** GP Puro
- **Precisión (RMSE):** 40.16

**Por qué este modelo:**

El sistema determinó que la capacidad del GP para capturar la tendencia suave superó a la reactividad del LightGBM en esta categoría estable.

---

### B. Desglose de la Orden (Los Números)

El número grande **(2,411 unidades)** no es una adivinanza, es una **ecuación de riesgo ajustada**.

#### **Demanda Base: 1,549 unidades**

Esta es la predicción de la Media (μ). Es lo que el modelo cree que probablemente se venderá. Si usáramos un ERP tradicional, compraríamos solo esto.

#### **Stock de Seguridad: +861 unidades**

Esta es la **protección contra la incertidumbre**.

**Fórmula:**
```
SS = Z_score × σ_GP × √T
```

**Insight:**

El Stock de Seguridad representa un **~55% adicional** sobre la base. Esto indica que, aunque la tendencia es estable, la variabilidad semanal es alta. El modelo sugiere protegerse agresivamente.

---

### C. Métricas Estructurales

#### **Memoria GP: 45.5 semanas**

Este valor (el Length Scale) indica una inercia de casi un año.

**Lectura:** Es una categoría **Estratégica**. Las decisiones de compra pueden hacerse a largo plazo.

#### **Rango Probable: 687 - 2,411 unidades**

El modelo admite que, en el peor escenario (baja demanda), podríamos vender solo 687 unidades.

---

### D. Interpretación Visual (Gráfico)

**Línea Azul:**
- Representa la tendencia central
- Nótese cómo sigue los puntos históricos (negros) ignorando el ruido extremo

**Sombra Azul:**
- Representa la zona de riesgo
- Su amplitud visualiza el "colchón" que estamos comprando con las unidades extra de seguridad

---

## 4. Matriz de Decisión: Trade-offs

La interfaz permite ajustar el **Nivel de Servicio** (Slider), lo que tiene implicaciones financieras directas:

| Nivel Seleccionado | Z-Score | Interpretación de Negocio | Cuándo usarlo |
|-------------------|---------|---------------------------|---------------|
| **80% (Bajo)** | 0.84 | "Acepto que 2 de cada 10 clientes no encuentren stock" | Productos de bajo margen o alta sustitución. Ahorra capital. |
| **95% (Estándar)** | 1.65 | "Estándar de la industria" | El equilibrio por defecto para la mayoría de productos. |
| **99% (Crítico)** | 2.33 | "Nadie se queda sin producto" | Productos VIP, lanzamientos o ítems de margen muy alto. Costoso en almacenamiento. |

---

## 💡 Conclusión

Esta herramienta permite a Olist pasar a un modelo de **"Gestión Financiera de Probabilidades"**.

**Valor agregado:**
- ✅ Cuantificación de incertidumbre en tiempo real
- ✅ Optimización de trade-off costo-servicio
- ✅ Predicciones adaptadas a Small Data
- ✅ Interfaz intuitiva para decisores no técnicos

---

### Interpretación de Resultados

**Panel de KPIs:**
- **Orden Sugerida:** Cantidad total a pedir (Demanda Base + Stock Seguridad)
- **Demanda Base:** Predicción esperada del modelo ganador
- **Stock Seguridad:** Buffer basado en incertidumbre (σ × Z-score)
- **Memoria GP:** Radio temporal de influencia detectado por el modelo

**Gráfico:**
- **Puntos negros:** Ventas históricas reales
- **Línea azul:** Predicción del modelo (tendencia)
- **Zona sombreada:** Rango de incertidumbre según nivel de servicio
- **Línea roja vertical:** Límite "Hoy" (histórico vs proyección)

**Ajuste Dinámico:**
El slider de "Horizonte" se ajusta automáticamente según la memoria predictiva:
- **Memoria alta (>40 sem):** Permite planificar hasta 52 semanas
- **Memoria baja (<20 sem):** Limita a 8-20 semanas (evita sobre-proyección)

---
# 🗺️ PARTE 6: INTELIGENCIA ESPACIAL (FASE 3)
---

## Objetivo

Predecir retrasos logísticos en **cualquier coordenada** de Brasil, incluso en ubicaciones sin historial de entregas.

**Pregunta de negocio:**
> "Si un cliente hace un pedido desde coordenadas (-12.15, -44.99) donde nunca hemos entregado,
> ¿cuál es el riesgo de retraso?"

---

## Metodología: Kriging Espacial

**Kriging** es una técnica de interpolación geoestadística que:

- ✅ **Modela la correlación espacial** entre puntos geográficos
- ✅ **Predice valores en ubicaciones no observadas** (interpolación inteligente)
- ✅ **Cuantifica la incertidumbre** de cada predicción

**Diferencia vs interpolación simple:**
```
Interpolación lineal: "Promedio entre los 2 puntos más cercanos"
Kriging: "Promedio ponderado de TODOS los puntos según su correlación espacial"
```

---

### Kernel RBF (Radial Basis Function)

Modelamos problemas logísticos como **"ondas expansivas"**:

**Concepto:**
- Un retraso en São Paulo **no afecta igual** a Rio de Janeiro (200 km) que a Manaus (2,500 km)
- El impacto es **máximo en el epicentro** y se desvanece con la distancia

**Fórmula:**
```
k(r) = σ² · exp(-r² / 2l²)

Donde:
- r = distancia entre dos puntos (en grados geográficos)
- l = length_scale = "radio de influencia" aprendido por el modelo
- σ² = varianza de la señal
```

**Parámetros aprendidos:**
- **Length Scale:** 2.96° ≈ **329 km**
  - **Interpretación:** Un problema logístico en una ciudad afecta a ~330 km a la redonda
  - Más allá de esta distancia, la correlación es débil

---

## Mapa de Calor Histórico

Visualización de **retrasos promedio** por coordenada basada en 4,000 puntos de entrega históricos.

**Leyenda de colores:**
- 🔵 **Azul:** Entregas rápidas (hasta 5 días antes de lo prometido)
- 🟢 **Verde:** Entregas puntuales (±2 días de lo prometido)
- 🟡 **Amarillo:** Retrasos moderados (3-7 días)
- 🔴 **Rojo:** Retrasos críticos (>7 días)

## Mapa de Calor Histórico

In [ ]:
# Guardar y mostrar mapa interactivo
# Los mapas HTML se guardan en la carpeta outputs/
# Haz doble clic en el archivo para abrirlo en tu navegador

# Si el mapa no se visualiza en el notebook, abre el archivo directamente:
import os
mapa_files = list(PROJECT_ROOT.glob('outputs/mapa_*.html'))
print("📍 Mapas disponibles:")
for f in mapa_files:
    print(f"   - {f.name}")
print("\n💡 Haz clic derecho > Abrir con > Tu navegador")


### Interpretación del Mapa Histórico

**Zonas críticas identificadas (rojo/naranja):**

1. **Nordeste (Recife, Salvador):**
   - Alta concentración de retrasos históricos
   - Posibles causas: Infraestructura limitada, distancia a centros de distribución

2. **Norte (Amazonas):**
   - Retrasos por desafíos geográficos (selva, ríos)
   - Acceso limitado por carretera

3. **Interior de Brasil:**
   - Zonas dispersas con entregas menos frecuentes
   - Menor inversión logística

**Zonas eficientes (azul/verde):**

1. **Eje São Paulo - Rio de Janeiro:**
   - Infraestructura desarrollada
   - Alta frecuencia de rutas

2. **Sur (Florianópolis, Curitiba):**
   - Geografía favorable
   - Buenos tiempos de entrega

**Limitación clave:**
Este mapa solo muestra **puntos con historial**. ¿Qué pasa en las zonas blancas (sin datos)?

**Solución:** Modelo de Kriging para **interpolación inteligente**.

---

## Entrenamiento del Modelo Espacial

### Datos utilizados

- **Puntos de entrenamiento:** 4,000 coordenadas únicas
- **Puntos con retraso:** 423 (10.6% del total)
- **Variable objetivo:** Retraso promedio en días

### Parámetros del Modelo GP Espacial
```
📊 Parámetros Aprendidos:
   Length Scale: 2.96° (~329 km)
   Radio de influencia logístico: ~330 km a la redonda
   
   Puntos entrenamiento: 4,000
   Puntos con retraso: 423
```

**Interpretación del Length Scale:**

- **329 km** es la distancia en la que la correlación espacial es significativa
- **Ejemplo práctico:**
  - Problema en São Paulo (lat: -23.55, lng: -46.63)
  - Afecta a Campinas (90 km) con alta probabilidad
  - Afecta a Rio de Janeiro (430 km) con baja probabilidad

**Validación:**
- Este radio es **consistente con redes logísticas regionales** en Brasil
- Coincide con áreas de distribución típicas de centros logísticos

---

## Mapa Maestro de Predicciones

Este mapa combina:

1. **Superficie interpolada** - Predicciones en toda Brasil (incluso sin historial)
2. **Transparencia adaptativa** - Zonas con alta incertidumbre son más transparentes
3. **Puntos de auditoría** - Datos reales superpuestos para validación
4. **Detección de outliers** - Puntos magenta = eventos aislados (no sistémicos)

**Características técnicas:**
- **Grid de predicción:** 50×50 puntos distribuidos uniformemente
- **Método:** Kriging Gaussiano con kernel RBF
- **Incertidumbre:** Calculada como desviación estándar (σ) de la distribución posterior

## Entrenamiento del Modelo Espacial

In [ ]:
# Cargar modelo espacial (con fallback a entrenamiento)
import joblib
from pathlib import Path

model_path = PROJECT_ROOT / 'checkpoints' / 'gp_spatial_model.pkl'

if model_path.exists():
    print("⚡ Cargando modelo espacial desde cache...")
    gp_spatial = joblib.load(model_path)
    print("   ✅ Modelo cargado")
else:
    print("⚠️ Modelo no encontrado en cache")
    print("🔄 Entrenando modelo espacial desde cero...")
    print("   (Esto tomará ~5-10 minutos)")

    # Entrenar desde cero
    from src.models.spatial_kriging import SpatialKriging

    # Preparar datos espaciales
    df_spatial = df_main[df_main['delay_days'].notna()].copy()

    # Entrenar (flujo correcto: aggregate -> sample -> train)
    kriging = SpatialKriging()
    kriging.aggregate_spatial(df_spatial)
    kriging.stratified_sampling()
    metrics = kriging.train()
    gp_spatial = kriging.model

    # Guardar
    joblib.dump(gp_spatial, model_path)
    print(f"   ✅ Modelo guardado en: {model_path}")

# Cargar datos geográficos
df_train_geo_path = PROJECT_ROOT / 'checkpoints' / 'df_spatial.parquet'

if df_train_geo_path.exists():
    df_train_geo = pd.read_parquet(df_train_geo_path)
else:
    df_train_geo = df_main[df_main['delay_days'].notna()].copy()
    df_train_geo.to_parquet(df_train_geo_path)


In [ ]:
# Guardar y mostrar mapa interactivo
# Los mapas HTML se guardan en la carpeta outputs/
# Haz doble clic en el archivo para abrirlo en tu navegador

# Si el mapa no se visualiza en el notebook, abre el archivo directamente:
import os
mapa_files = list(PROJECT_ROOT.glob('outputs/mapa_*.html'))
print("📍 Mapas disponibles:")
for f in mapa_files:
    print(f"   - {f.name}")
print("\n💡 Haz clic derecho > Abrir con > Tu navegador")


### Interpretación del Mapa Maestro

#### **Colores de la superficie (fondo):**

- 🟢 **Verde:** Bajo riesgo de retraso (-16.6 días = entregas anticipadas)
- 🟡 **Amarillo:** Riesgo moderado (entregas puntuales)
- 🟠 **Naranja:** Alto riesgo (3-5 días de retraso)
- 🔴 **Rojo:** Riesgo crítico (>5 días de retraso)

**Escala:** -16.6d (Certeza Alta) → -5.3d (Niebla)

#### **Transparencia del fondo:**

- **Opaco:** Alta confianza (muchos datos históricos cercanos)
- **Translúcido:** Baja confianza (zona interpolada, pocas entregas históricas)

**Ejemplo:**
- São Paulo (opaco verde) → Predicción muy confiable
- Interior del Amazonas (translúcido) → Predicción con alta incertidumbre

#### **Puntos superpuestos (auditoría):**

- ⚫ **Negro:** Entregas normales (dentro de lo esperado)
- 🟣 **Magenta:** Retrasos críticos (> -4.2 días)

#### **Outliers Espaciales (puntos magenta en zonas verdes):**

**¿Qué significan?**
- Eventos aislados que **NO representan el patrón de la zona**
- Ejemplos: Huelga puntual, accidente en carretera, error operativo

**Por qué el modelo los ignora:**
- El Kriging prioriza **tendencias regionales** sobre ruido local
- Decisión correcta: No queremos que un evento único distorsione la predicción

**Validación:**
- Si vemos muchos puntos magenta en una zona verde → Revisar modelo
- Si son pocos y dispersos → Confirma que el modelo es robusto

---

## Casos de Uso Prácticos

### **Caso 1: Expansión Geográfica**

**Pregunta:** ¿Deberíamos abrir operaciones en Barreiras, Bahía (-12.15, -44.99)?

**Análisis con el mapa:**
1. Buscar coordenadas en el mapa maestro
2. Color observado: Verde/Amarillo (riesgo bajo-moderado)
3. Transparencia: Media (confianza moderada)

**Conclusión:**
- ✅ Zona geográficamente favorable
- ⚠️ Poca historia de entregas → Monitoreo inicial necesario
- 💡 Recomendación: Piloto con SLA conservador (no prometer entrega rápida)

---

### **Caso 2: Priorización de Inversión en Infraestructura**

**Pregunta:** ¿Dónde invertir en nuevos centros de distribución?

**Análisis:**
1. Identificar zonas rojas/naranjas con alta opacidad (problema sistémico confirmado)
2. Calcular volumen de pedidos en esas zonas
3. Estimar ROI de reducir retrasos

**Ejemplo del mapa:**
- **Nordeste (Recife, Salvador):** Zona roja, alta densidad de puntos negros
  - **Diagnóstico:** Problema estructural, no eventos aislados
  - **Acción:** Considerar hub regional en Salvador

- **Interior de Minas Gerais:** Zona amarilla, baja densidad
  - **Diagnóstico:** Pocos pedidos, no justifica inversión
  - **Acción:** Mantener entregas desde São Paulo con SLA flexible

---


---

## Limitaciones y Consideraciones

### **Lo que el modelo SÍ detecta:**

✅ Patrones regionales históricos persistentes  
✅ Zonas geográficamente difíciles (infraestructura limitada)  
✅ Correlación espacial entre ciudades cercanas  

### **Lo que el modelo NO detecta:**

❌ Eventos en tiempo real (accidentes, huelgas actuales)  
❌ Clima extremo momentáneo (lluvias, nevadas)  
❌ Problemas de acceso específicos (calles cerradas)  

**Implicación:**
- El modelo da **predicción base** (tendencia histórica)
- Debe complementarse con **datos en tiempo real** para decisiones operativas

---

### **Caveats Técnicos:**

1. **Length Scale (329 km) es un promedio:**
   - En zonas urbanas densas (São Paulo), la correlación puede ser de 50-100 km
   - En zonas rurales, puede extenderse hasta 500 km

2. **Outliers magenta:**
   - No son "errores" del modelo
   - Representan eventos únicos que no deben influir en la tendencia general

3. **Transparencia como proxy de incertidumbre:**
   - No es métrica exacta de σ
   - Es visualización intuitiva para decisores no técnicos

---

## Conclusiones de la Fase Espacial

### **Valor Agregado del Sistema:**

| Capacidad | Sistema Tradicional | Sistema con Kriging |
|-----------|---------------------|---------------------|
| **Predicción en zonas nuevas** | ❌ No disponible | ✅ Interpolación inteligente |
| **Cuantificación de riesgo** | ❌ Binario (sí/no cobertura) | ✅ Gradiente de probabilidad |
| **Ajuste dinámico de ETA** | ❌ ETA fijo por estado | ✅ ETA por coordenada exacta |
| **Identificación de outliers** | ❌ No distingue eventos aislados | ✅ Separa ruido de patrón |

### **Impacto de Negocio:**

**Escenario actual (sin modelo):**
- Cliente en zona sin historial → "No hacemos entregas" o "ETA genérico de 15 días"
- Pérdida de ventas potenciales o promesas poco confiables

**Escenario con modelo:**
- Cliente en misma zona → Modelo predice riesgo moderado (σ = 3 días)
- ETA personalizado: 8 días ± 3 (rango: 5-11 días)
- **Resultado:** Venta cerrada con promesa realista

**ROI estimado:**
- Reducción de promesas incumplidas: ~15%
- Aumento en conversión en zonas "grises": ~8%
- Ahorro en compensaciones por retraso: ~$12,000 USD/año

---



---
# 📍 PARTE 6.5: CALCULADORA DE ENVÍOS (SPOT-CHECK)
---

## Herramienta Táctica de Validación

Una vez modelada la superficie de riesgo de todo Brasil, operacionalizamos este conocimiento mediante una **Interfaz de Consulta Puntual**.

Esta herramienta permite a los equipos de:
- **Atención al Cliente:** Evaluar viabilidad de entregas en direcciones específicas
- **Logística:** Calcular ETAs ajustados por zona geográfica
- **Ventas:** Prometer fechas realistas basadas en ubicación exacta

## Lógica del Algoritmo

**1. Inferencia Gaussiana:** Al ingresar coordenadas (Lat, Lng), el modelo Kriging interpola el retraso esperado y la incertidumbre (σ), incluso en puntos sin historial.

**2. Geocoding Inverso:** Busca la ciudad más cercana en la base histórica para dar contexto.

**3. Calibración de Riesgo:**
- **Semáforo de Desempeño:** Compara predicción vs promesa (Anticipación vs Retraso)
- **Semáforo de Confianza:** Evalúa distancia a datos conocidos

## Guía de Interpretación

**Pronóstico (Color de Fondo):**
- 🟢 **Verde (Anticipada):** El paquete llegará ANTES de la fecha límite (Margen de seguridad)
- 🔴 **Rojo (Retraso):** El paquete llegará DESPUÉS de la fecha límite (Añadir días extra al ETA)

**Certeza del Cálculo:**
- **ALTA:** Muchos datos históricos → Predicción muy confiable
- **MEDIA:** Inferencia regional → Confianza moderada
- **BAJA:** Zona con poca información → Cautela recomendada

## Contexto Industrial

**Lo que el modelo SÍ detecta:**
- Patrones regionales históricos
- Zonas geográficamente difíciles
- Errores en tablas estáticas de ETA

**Lo que el modelo NO detecta:**
- Eventos en tiempo real (accidentes, huelgas)
- Clima extremo momentáneo
- Problemas de acceso específicos

**Propuesta de valor vs sistemas tradicionales:**
1. **GPS exacto** vs código postal (corrección de frontera administrativa)
2. **Smart buffering dinámico** por zona
3. **Semáforo de incertidumbre** (el sistema tradicional es ciego a su ignorancia)

In [ ]:
# Cargar modelo espacial (con fallback a entrenamiento)
import joblib
from pathlib import Path

model_path = PROJECT_ROOT / 'checkpoints' / 'gp_spatial_model.pkl'

if model_path.exists():
    print("⚡ Cargando modelo espacial desde cache...")
    gp_spatial = joblib.load(model_path)
    print("   ✅ Modelo cargado")
else:
    print("⚠️ Modelo no encontrado en cache")
    print("🔄 Entrenando modelo espacial desde cero...")
    print("   (Esto tomará ~5-10 minutos)")

    # Entrenar desde cero
    from src.models.spatial_kriging import SpatialKriging

    # Preparar datos espaciales
    df_spatial = df_main[df_main['delay_days'].notna()].copy()

    # Entrenar (flujo correcto: aggregate -> sample -> train)
    kriging = SpatialKriging()
    kriging.aggregate_spatial(df_spatial)
    kriging.stratified_sampling()
    metrics = kriging.train()
    gp_spatial = kriging.model

    # Guardar
    joblib.dump(gp_spatial, model_path)
    print(f"   ✅ Modelo guardado en: {model_path}")

# Cargar datos geográficos
df_train_geo_path = PROJECT_ROOT / 'checkpoints' / 'df_spatial.parquet'

if df_train_geo_path.exists():
    df_train_geo = pd.read_parquet(df_train_geo_path)
else:
    df_train_geo = df_main[df_main['delay_days'].notna()].copy()
    df_train_geo.to_parquet(df_train_geo_path)


---
# 📍 PARTE 6.5: CALCULADORA DE ENVÍOS (SPOT-CHECK)
---

## Herramienta Táctica de Validación

Una vez modelada la superficie de riesgo de todo Brasil mediante Kriging espacial,
operacionalizamos este conocimiento mediante una **Interfaz de Consulta Puntual**.

Esta herramienta permite a diferentes equipos tomar decisiones informadas en tiempo real:

| Equipo | Uso de la Calculadora |
|--------|----------------------|
| **Atención al Cliente** | Evaluar viabilidad de entregas en direcciones específicas |
| **Logística** | Calcular ETAs ajustados por zona geográfica |
| **Ventas** | Prometer fechas realistas basadas en ubicación exacta |
| **Expansión** | Analizar nuevas zonas de cobertura antes de invertir |

---

## 🔬 Lógica del Algoritmo

### **1. Inferencia Gaussiana**

Al ingresar coordenadas (Lat, Lng), el modelo Kriging realiza **interpolación espacial**:
```
Input: (-23.55, -46.63)  [São Paulo]
       ↓
Modelo GP Espacial (Kriging)
       ↓
Output: μ = -10.3 días (anticipación)
        σ = 2.1 días (incertidumbre)
```

**Clave:** Funciona **incluso en puntos sin historial** de entregas.

---

### **2. Geocoding Inverso**

Busca la ciudad más cercana en la base histórica para dar **contexto geográfico**:
```python
# Pseudocódigo
def encontrar_ciudad_cercana(lat, lng):
    distancias = calcular_distancias(lat, lng, ciudades_conocidas)
    ciudad_mas_cercana = min(distancias)
    return ciudad_mas_cercana, distancia

# Ejemplo:
# Input: (-23.50, -46.70)
# Output: "Sao Paulo" (distancia: 8.59 km)
```

**Beneficio:** El usuario recibe contexto humano ("Cerca de São Paulo") además de números.

---

### **3. Calibración de Riesgo (Dual-Semaphore System)**

El sistema evalúa **dos dimensiones** independientes:

#### **A. Semáforo de Desempeño (Predicción vs Promesa)**
```
Predicción: -10.3 días (llega 10 días antes)
Promesa estándar: 15 días
       ↓
Análisis: -10.3 < 0 (anticipación)
       ↓
Semáforo: 🟢 VERDE (Anticipada)
```

**Criterios:**
- 🟢 **Verde:** Predicción < 0 (entrega anticipada)
- 🔴 **Rojo:** Predicción > 0 (retraso esperado)

---

#### **B. Semáforo de Confianza (Distancia a Datos)**
```
Distancia a ciudad conocida: 8.59 km
Threshold ALTA: < 10 km
       ↓
Semáforo: ✅ ALTA
```

**Criterios:**
- ✅ **ALTA:** Distancia < 10 km (muchos datos históricos)
- ⚠️ **MEDIA:** Distancia 10-50 km (inferencia regional)
- ❌ **BAJA:** Distancia > 50 km (poca información)

**Valor diferencial:**
> Los sistemas tradicionales dan un número sin decirte **qué tan confiable es**.
> Nuestro sistema admite su propia ignorancia mediante el semáforo de confianza.

---

## 📊 Guía de Interpretación

### **Pronóstico (Color de Fondo del Panel)**

| Color | Significado | Interpretación | Acción Recomendada |
|-------|-------------|----------------|-------------------|
| 🟢 **Verde** | Anticipada | Paquete llegará ANTES de la fecha límite | Prometer entrega rápida con confianza |
| 🔴 **Rojo** | Retraso | Paquete llegará DESPUÉS de la fecha límite | Ajustar ETA + días extra |

**Ejemplo (São Paulo):**
- Predicción: **-10.3 días**
- Fondo: 🟢 **Verde**
- Mensaje: "ANTICIPADA"
- **Decisión:** Podemos prometer entrega en 5 días (con margen de 10 días)

---

### **Certeza del Cálculo**

| Nivel | Significado | Cuándo usar la predicción | Precauciones |
|-------|-------------|---------------------------|-------------|
| ✅ **ALTA** | Muchos datos históricos | Confiar plenamente en predicción | Ninguna |
| ⚠️ **MEDIA** | Inferencia regional | Usar con buffer adicional | Monitorear primeras entregas |
| ❌ **BAJA** | Zona con poca información | Solo referencia aproximada | No comprometer SLA agresivo |

**Ejemplo (São Paulo):**
- Certeza: ✅ **ALTA**
- σ = 2.1 días
- Distancia a datos: 8.59 km
- **Decisión:** Predicción muy confiable, usar sin ajustes

---

## 🌐 Contexto Industrial

### **Lo que el modelo SÍ detecta:**

✅ **Patrones regionales históricos persistentes**
- Ejemplo: Nordeste históricamente más lento que Sur

✅ **Zonas geográficamente difíciles**
- Ejemplo: Amazonas (infraestructura limitada)

✅ **Errores en tablas estáticas de ETA**
- Ejemplo: ERP promete 7 días en zona que históricamente toma 12

---

### **Lo que el modelo NO detecta:**

❌ **Eventos en tiempo real**
- Accidentes en carreteras hoy
- Huelgas de transportistas actuales

❌ **Clima extremo momentáneo**
- Lluvias torrenciales esta semana
- Nevadas en sur de Brasil

❌ **Problemas de acceso específicos**
- Calle cerrada por obras
- Edificio sin acceso para camiones

**Implicación:**
- Modelo da **tendencia histórica** (baseline)
- Debe complementarse con **datos operativos** para decisiones del día

---

## 🎯 Propuesta de Valor vs Sistemas Tradicionales

| Aspecto | Sistema Tradicional (ERP) | Nuestra Calculadora |
|---------|--------------------------|---------------------|
| **Granularidad** | Por código postal (5 dígitos) | Por coordenadas GPS exactas |
| **Incertidumbre** | ❌ No cuantificada | ✅ Semáforo de confianza (σ) |
| **Zonas nuevas** | ❌ "No hay cobertura" | ✅ Interpolación inteligente |
| **Fronteras administrativas** | ❌ Error en límites de CEP | ✅ Corrige con GPS real |
| **Buffer dinámico** | ❌ Fijo (ej. +3 días siempre) | ✅ Ajustado por zona (σ variable) |

### **1. GPS Exacto vs Código Postal**

**Problema del CEP (código postal):**
```
CEP 01310-100 (São Paulo centro)
  vs
CEP 01310-900 (límite con barrio lejano)

Mismo CEP, realidad logística MUY diferente
```

**Nuestra solución:**
- Coordenadas exactas (-23.55, -46.63)
- Interpolación espacial continua (no saltos en fronteras)

---

### **2. Smart Buffering Dinámico**

**ERP tradicional:**
```
ETA = Días base + 3 días (buffer fijo para todos)
```

**Nuestra calculadora:**
```
ETA = μ + (Z-score × σ)

São Paulo: μ=-10.3, σ=2.1 → ETA = 5 días (confianza alta)
Amazonas: μ=+5.2, σ=8.3 → ETA = 15 días (alta incertidumbre)
```

**Resultado:**
- ✅ No desperdiciamos buffer en zonas eficientes
- ✅ Protegemos más en zonas volátiles

---

### **3. Semáforo de Incertidumbre**

**Sistema tradicional:**
```
"ETA: 7 días"

¿Qué tan confiable es? 🤷 No sabemos
```

**Nuestra calculadora:**
```
"ETA: 7 días"
Certeza: ALTA (σ: 2.1)
       vs
"ETA: 7 días"  
Certeza: BAJA (σ: 9.8)

Ahora sabemos CUÁNTO confiar en la predicción
```

---

## 💼 Caso de Estudio: Barreiras, Bahía

### **Coordenadas de prueba:** (-12.15, -44.99)

**Resultado típico:**
```
🚀 Predicción: -9.2 días (anticipación)
📍 Ciudad cercana: Barreiras (distancia: 15 km)
⚠️ Certeza: MEDIA (σ: 6.5 días)
💡 Recomendación: "Es seguro prometer entrega rápida. Holgura: 9.2 días."
```

---

### **Interpretación:**

#### **✅ Lo que el modelo SÍ sabe:**

- La **región general** (oeste de Bahía) tiende a ser eficiente
- Patrones de ciudades cercanas (Barreirinhas, Luís Eduardo Magalhães)
- Correlación espacial con zonas similares

#### **⚠️ Lo que el modelo admite NO saber:**

- No tenemos **datos densos** en esa coordenada exacta
- σ = 6.5 es **moderadamente alto** (vs σ = 2.1 en São Paulo)
- Distancia a ciudad conocida: 15 km (threshold MEDIA)

---

### **Decisión de Negocio:**

| Escenario | Acción |
|-----------|--------|
| **Venta normal** | ✅ Prometer entrega en 7-10 días (con margen) |
| **Cliente VIP** | ⚠️ No prometer <5 días (riesgo moderado) |
| **Primera entrega en zona** | 🔍 Track & Trace intensivo (validar predicción) |
| **Expansión a largo plazo** | 📊 Monitorear primeras 20 entregas, recalibrar |

---

### **Por qué este ejemplo es poderoso:**

> "Este es un ejemplo perfecto de cómo el **semáforo de incertidumbre** (σ) es
> el verdadero diferenciador. Los sistemas tradicionales son **ciegos a su propia ignorancia**."

**Sistema tradicional:**
```
Input: CEP de Barreiras
Output: "ETA: 10 días"

¿Confiable? No sabemos
¿De dónde sale? Tabla estática
¿Qué tan arriesgado? 🤷
```

**Nuestra calculadora:**
```
Input: (-12.15, -44.99)
Output: "ETA: 10 días"
        "Certeza: MEDIA (σ=6.5)"
        "Interpolación de zona vecina (15 km)"

Ahora tenemos CONTEXTO para decidir
```

---

## 🚀 Demostración Interactiva

A continuación, la interfaz permite probar predicciones en diferentes zonas de Brasil.

## 📦 Calculadora de Envíos

Herramienta de evaluación de riesgo geográfico que utiliza modelos espaciales Kriging para predecir retrasos de entrega en cualquier coordenada de Brasil.

<img src="../outputs/Calculadora%20de%20envios.png" width="100%">

**Funcionalidades:**
- 🗺️ Predicción de retrasos por ubicación geográfica
- 📍 Compatible con cualquier coordenada de Brasil
- 📊 Visualización de incertidumbre espacial
- 💡 Recomendaciones de logística basadas en datos

In [ ]:
# Setup path
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Forzar recarga del módulo
import importlib
if 'src.apps.shipping_calculator' in sys.modules:
    importlib.reload(sys.modules['src.apps.shipping_calculator'])

# Lanzar calculadora de envíos
from src.apps.shipping_calculator import ShippingCalculator

print("🚀 Inicializando calculadora de envíos...")
print("⏳ Esto puede tomar 30-60 segundos (calibración automática)...\n")

calc = ShippingCalculator(gp_spatial, df_main)
calc.render()

print("\n✅ Calculadora lista")
print("\n💡 Coordenadas de ejemplo para probar:")
print("   • São Paulo: -23.55, -46.63")
print("   • Rio de Janeiro: -22.91, -43.17")
print("   • Brasília: -15.79, -47.88")
print("   • Manaus (Amazonas): -3.11, -60.02")
print("   • Barreiras (Bahía): -12.15, -44.99")

---
# 📋 PARTE 7: CONCLUSIONES Y RECOMENDACIONES
---

## Resultados Principales

### Forecasting Temporal (Fase 2):

✅ **No hay modelo universal superior**
- Prueba de Friedman: p=0.1468 > 0.05
- Distribución equilibrada: GP (33%), LGBM (33%), MA (33%)
- Necesidad de selección por categoría

✅ **Mejora promedio: 15.5%** vs baselines
- Mejor: furniture_decor (+32.4%)
- housewares: RMSE = 26.33 (categoría más predecible)

✅ **Insights críticos:**
- Híbrido (GP+LGBM) falló completamente (0/6 victorias)
- Baseline ganó 2 categorías (volatilidad extrema)
- CV reveló inestabilidades ocultas en single-split

### Inteligencia Espacial (Fase 3):

✅ **Radio de influencia: ~111 km**
- Problemas logísticos son regionales, no locales
- Correlación espacial significativa

✅ **Capacidad predictiva:**
- Interpolación exitosa en zonas sin datos
- Incertidumbre cuantificada (transparencia)
- Detección de outliers espaciales

✅ **Impacto operacional:**
- ETAs dinámicos por coordenada
- Alertas tempranas en zonas de riesgo
- Optimización de rutas

## Recomendaciones

### Corto Plazo (0-3 meses):
1. Implementar modelo selector en producción
2. Ajustar políticas de stock por categoría
3. Integrar predicciones espaciales en sistema ETA

### Mediano Plazo (3-6 meses):
1. A/B testing de estrategias de inventario
2. Sistema de alertas geográficas
3. Dashboard en tiempo real

### Largo Plazo (6-12 meses):
1. Drift detection automático
2. Explicabilidad con SHAP
3. Multi-echelon inventory optimization
4. Integración con pricing dinámico

## Limitaciones y Trabajo Futuro

**Limitaciones:**
- Dataset histórico (2016-2018)
- No incluye factores externos (clima, eventos)
- Single-echelon (no considera múltiples warehouses)

**Próximos pasos:**
- [ ] MLOps pipeline (CI/CD)
- [ ] Reentrenamiento automático
- [ ] API REST para predicciones
- [ ] Containerización (Docker)
- [ ] Explicabilidad (SHAP/LIME)

## Arquitectura vs Industria

| Aspecto | Amazon/Walmart | Este Proyecto |
|---------|----------------|---------------|
| Modelos Ensemble | ✅ | ✅ |
| Time Series CV | ✅ | ✅ |
| Statistical Tests | ✅ | ✅ |
| Spatial Analytics | ✅ | ✅ |
| Auto Model Selection | ML-based | Rule-based |
| MLOps | ✅ | ⏳ Roadmap |
| Drift Detection | ✅ | ⏳ Roadmap |

**Conclusión:** Este proyecto implementa metodologías de nivel profesional,
con gaps identificados para evolución a sistema production-grade.

---
# 🏗️ PARTE 7: OPTIMIZACIÓN DE RED (FACILITY LOCATION)
---

## Desafío de Negocio

El **Mapa de Riesgo Espacial** (Fase 3) reveló "manchas rojas" críticas, especialmente
en Norte/Nordeste, donde la promesa de entrega falla sistemáticamente.

**Problema identificado:**
- Atender estas zonas desde almacenes actuales (São Paulo/Sur) es **ineficiente** debido a la distancia
- Retrasos promedio >10 días en regiones alejadas
- Tasa de incumplimiento de promesas >25% en zonas críticas

**Solución propuesta:**
- Abrir **Hubs de Tránsito satélite** ubicados estratégicamente
- Reducir distancia promedio a zonas problemáticas
- Mejorar cobertura geográfica sin saturar hubs principales

---

## Metodología: K-Means Ponderado por "Dolor Logístico"

### Filosofía del Enfoque

En lugar de ubicar hubs donde hay **más ventas** (enfoque tradicional), los ubicamos
donde hay **más dolor** acumulado.

**Comparación:**
```
❌ Enfoque tradicional (por volumen de ventas):
   → Resultado: Hubs concentrados en São Paulo, Rio, Belo Horizonte
   → Problema: Ya funcionamos bien ahí, no resuelve las zonas críticas

✅ Nuestro enfoque (por dolor acumulado):
   → Resultado: Hubs en zonas con retrasos sistemáticos
   → Beneficio: Ataca directamente el problema
```

---

### Algoritmo: K-Means con Ponderación Cuadrática

**Paso 1: Filtrar datos**
```python
df_pain = df_main[df_main['delay_days'] > 0]
```
- Solo analizamos pedidos **con retraso** (ignora entregas exitosas)
- Razón: No queremos poner hubs donde ya funciona bien

**Paso 2: Asignar pesos**
```python
weights = delay_days ** 2.0
```
- Cada pedido retrasado tiene un "peso" = (días de retraso)²
- Ejemplos:
  - Retraso de 3 días → peso = 9
  - Retraso de 10 días → peso = 100

**¿Por qué elevar al cuadrado?**
- Penaliza **desproporcionadamente** los retrasos largos
- Un retraso de 10 días pesa **11 veces más** que uno de 3 días
- Enfoca recursos donde el problema es **grave**, no solo frecuente

**Paso 3: Ejecutar K-Means**
```python
kmeans.fit(coordenadas, sample_weight=weights)
```
- K-Means busca `n_hubs` centroides
- Pedidos con mucho peso "jalan" más fuerte los centroides
- Resultado: Hubs ubicados en "centros de gravedad del dolor"

**Parámetros técnicos:**
- `n_clusters=4` → Número de hubs a proponer
- `random_state=42` → Reproducibilidad
- `n_init=10` → Ejecuta 10 veces y elige la mejor solución

---

In [ ]:
# ============================================================================
# OPTIMIZACIÓN DE UBICACIÓN DE HUBS
# ============================================================================

import sys
from pathlib import Path

project_root = Path(str(PROJECT_ROOT))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Forzar recarga
import importlib
if 'src.optimization.facility_location' in sys.modules:
    importlib.reload(sys.modules['src.optimization.facility_location'])

print("="*80)
print("🏗️ OPTIMIZACIÓN DE RED LOGÍSTICA")
print("="*80)

from src.optimization.facility_location import FacilityLocator

# Inicializar
locator = FacilityLocator(df_main)

print("\n⚙️ Ejecutando K-Means ponderado...")
print(f"   Dataset: {df_main.shape[0]:,} órdenes totales")
print(f"   Parámetros: n_hubs=4, weight_power=2.0")

# Optimizar
results = locator.optimize_locations(n_hubs=4, weight_power=2.0)

print("\n" + "="*80)
print("📊 RESULTADOS DE LA OPTIMIZACIÓN")
print("="*80)

# Mostrar reporte
print(locator.get_summary_report(results))

# Guardar para análisis
print("\n💾 Guardando resultados...")
import pickle
with open(project_root / 'checkpoints' / 'facility_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print("   ✅ Guardado en: checkpoints/facility_results.pkl")

## 📊 Interpretación de Resultados

### Contexto General

**Datos clave:**
- **Total dataset:** 109,908 órdenes
- **Órdenes con retraso:** 7,239 (6.6% del total)
- **Estrategia:** Optimizar ubicación para atacar este 6.6% problemático

> **Nota importante:** Los porcentajes (66.3%, 15.7%, etc.) son del total de **órdenes retrasadas** (7,239),
> NO del total de órdenes (109,908). Es decir, estamos analizando dónde se concentra el problema.

---

### 🏭 HUB #1: Baependi, MG

**Ubicación:** (-21.96, -44.89) - Sur de Minas Gerais  
**Zona de influencia:** Interior de MG, SP, RJ  

#### Métricas
- **Impacto:** 4,796 órdenes retrasadas
- **% del problema total:** 66.3% de los retrasos
- **Retraso promedio actual:** 10.2 días

#### Análisis

**¿Por qué el algoritmo eligió este punto?**

1. **Centro de gravedad del Sur/Sudeste con problemas:**
   - Aunque São Paulo/Rio tienen buen servicio en zonas urbanas
   - El **interior** (zonas rurales de MG/SP) tiene alta dispersión
   - Baependi queda en posición estratégica entre estos interiores

2. **Ponderación cuadrática funcionando:**
   - Esta zona tiene volumen MODERADO pero retrasos ALTOS
   - Los 10.2 días promedio → peso = 10.2² = 104 por orden
   - El algoritmo priorizó "dolor" sobre "cantidad"

3. **66.3% del problema concentrado:**
   - Indica que la mayoría de retrasos están en una zona geográfica cohesiva
   - NO dispersos aleatoriamente
   - Un solo hub puede atacar la mayoría del problema

#### Recomendación de Negocio

**Tipo de hub:** Cross-docking (consolidación, no almacenamiento completo)

**Funciones:**
- Recibir carga de São Paulo/Campinas
- Consolidar entregas para interior de MG/SP/RJ
- Reducir distancia last-mile

**Prioridad:** 🔴 **CRÍTICA** (66% del dolor total)


---

### 🏭 HUB #2: Belém do São Francisco, PE

**Ubicación:** (-8.69, -38.91) - Interior de Pernambuco  
**Zona de influencia:** Nordeste (PE, BA, AL)  

#### Métricas
- **Impacto:** 1,138 órdenes retrasadas
- **% del problema total:** 15.7%
- **Retraso promedio actual:** 12.4 días (el MÁS ALTO)

#### Análisis

**¿Por qué este punto?**

1. **Epicentro del problema del Nordeste:**
   - Retraso de **12.4 días** es el peor de los 4 hubs
   - Peso cuadrático: 12.4² = 154 por orden
   - Aunque hay menos órdenes (1,138), el dolor es intenso

2. **Infraestructura limitada:**
   - Carreteras en mal estado
   - Pocos centros de distribución en la región
   - Distancia a São Paulo/Rio >2,000 km

3. **15.7% del problema en zona compacta:**
   - Indica problema regional sistémico
   - No son eventos aislados

#### Recomendación de Negocio

**Tipo de hub:** Regional con stock básico

**Funciones:**
- Almacenar productos de alta rotación local
- Coordinación con transportistas locales especializados
- Buffer para reducir dependencia de Sur/Sudeste

**Prioridad:** 🟡 **ALTA** (alto retraso, volumen moderado)


---

### 🏭 HUB #3: Santo Expedito do Sul, RS

**Ubicación:** (-27.95, -51.63) - Interior de Rio Grande do Sul  
**Zona de influencia:** Sur (RS, SC interior)  

#### Métricas
- **Impacto:** 999 órdenes retrasadas
- **% del problema total:** 13.8%
- **Retraso promedio actual:** 9.3 días

#### Análisis

**¿Por qué este punto?**

1. **Interior del Sur:**
   - Porto Alegre/Florianópolis tienen buen servicio
   - Pero interior (zona de serras, geografía complicada) sufre retrasos
   - Santo Expedito está en zona montañosa

2. **Retraso moderado pero volumen significativo:**
   - 9.3 días → peso = 86 por orden
   - Casi 1,000 órdenes representan 13.8% del problema

3. **Zona cohesiva:**
   - Problema no está disperso
   - Un hub puede cubrir eficientemente la zona

#### Recomendación de Negocio

**Tipo de hub:** Compartido/Colaborativo

**Funciones:**
- Hub compartido con otros operadores logísticos
- Reduce costos fijos
- Especializado en zonas de difícil acceso (serras)

**Prioridad:** 🟢 **MEDIA** (retraso moderado, volumen significativo)


---

### 🏭 HUB #4: Vitória do Jari, AP

**Ubicación:** (-1.51, -51.94) - Amapá (Norte de Brasil)  
**Zona de influencia:** Amazonas, Amapá, Pará  

#### Métricas
- **Impacto:** 306 órdenes retrasadas
- **% del problema total:** 4.2%
- **Retraso promedio actual:** 11.5 días

#### Análisis

**¿Por qué este punto?**

1. **Norte de Brasil - Zona más desafiante:**
   - Acceso limitado por carretera
   - Transporte fluvial necesario en muchas zonas
   - Infraestructura mínima

2. **Bajo volumen, alto retraso:**
   - Solo 306 órdenes (4.2% del problema)
   - Pero 11.5 días promedio → peso = 132 por orden
   - Algoritmo lo detectó por gravedad del retraso

3. **Dispersión geográfica extrema:**
   - Amazonas es ~5 veces el tamaño de São Paulo
   - Un solo hub no resuelve toda la región

#### Recomendación de Negocio

**⚠️ EVALUACIÓN CRÍTICA:**

**Opción A - NO abrir hub propio:**
- Costo de hub ($200k+) vs 306 órdenes/periodo
- ROI negativo en 3-5 años
- **Alternativa:** Contratos con operadores locales especializados (ej. Correios, LATAM Cargo)

**Opción B - Abrir hub SI:**
- El volumen crece significativamente
- Estrategia de expansión agresiva al Norte
- Alianzas con otros e-commerce para compartir costos

**Prioridad:** 🔵 **BAJA** (evaluar viabilidad económica antes de ejecutar)

**Recomendación:** Postponer decisión, monitorear crecimiento 6-12 meses

---

## 📈 Resumen Ejecutivo

| Hub | Zona | % Problema | Retraso | Prioridad | Inversión | Decisión |
|-----|------|-----------|---------|-----------|-----------|----------|
| #1 Baependi | MG Interior | 66.3% | 10.2d | 🔴 Crítica | $150k | ✅ Ejecutar |
| #2 Belém SF | Nordeste | 15.7% | 12.4d | 🟡 Alta | $200k | ✅ Ejecutar |
| #3 S. Expedito | Sur Interior | 13.8% | 9.3d | 🟢 Media | $100k | ✅ Ejecutar |
| #4 V. do Jari | Norte | 4.2% | 11.5d | 🔵 Baja | $200k+ | ⏸️ Evaluar |

**Cobertura con 3 hubs (excluir #4):** 95.8% del problema resuelto



**Impacto esperado:** Reducción de retrasos ~45% en zonas críticas

---

In [ ]:
# Guardar y mostrar mapa interactivo
# Los mapas HTML se guardan en la carpeta outputs/
# Haz doble clic en el archivo para abrirlo en tu navegador

# Si el mapa no se visualiza en el notebook, abre el archivo directamente:
import os
mapa_files = list(PROJECT_ROOT.glob('outputs/mapa_*.html'))
print("📍 Mapas disponibles:")
for f in mapa_files:
    print(f"   - {f.name}")
print("\n💡 Haz clic derecho > Abrir con > Tu navegador")


---

## 🗺️ Interpretación del Mapa Interactivo

### Elementos Visuales

**Heatmap de fondo (colores):**
- 🔵 **Azul:** Zonas de bajo riesgo (entregas eficientes)
- 🟢 **Verde:** Riesgo moderado
- 🔴 **Rojo:** Zonas críticas con retrasos sistemáticos

**Marcadores negros (🏭):**
- Ubicaciones óptimas propuestas por K-Means ponderado
- Posicionadas en "centros de gravedad del dolor logístico"

**Círculos punteados (⭕):**
- Radio de cobertura de **300 km**
- Distancia típica de distribución regional en Brasil
- Define el área de influencia de cada hub

---

### Análisis Geográfico por Hub

#### 🏭 **Hub #1: Baependi, MG (Centro-sur)**

**Observación en el mapa:**
- Está en zona **verde/amarilla** (no roja)
- ¿Por qué? Porque es el **centro de gravedad** de muchas zonas con problemas dispersos

**Interpretación:**
- NO está en la zona MÁS roja individual
- Está en la posición que **minimiza distancia total ponderada** a todos los problemas del Sur/Sudeste
- Su círculo de 300 km cubre: Interior de MG, norte de SP, sur de RJ

**Validación del algoritmo:**
- ✅ Correcto: No va al extremo (sería ineficiente para el resto)
- ✅ Busca posición balanceada para cubrir dispersión

---

#### 🏭 **Hub #2: Belém do São Francisco, PE (Nordeste)**

**Observación en el mapa:**
- Está en zona **verde** con manchas rojas cercanas
- Ubicado estratégicamente entre Pernambuco y Bahía

**Interpretación:**
- Centro de gravedad del Nordeste interior
- Su círculo cubre zonas problemáticas de PE, BA, AL
- Posición intermedia entre costa (Recife) e interior (sertão)

**Validación:**
- ✅ Evita extremos costeros (ya tienen mejor servicio)
- ✅ Enfoca en interior donde está el problema real

---

#### 🏭 **Hub #3: Santo Expedito do Sul, RS (Sur)**

**Observación en el mapa:**
- Está en zona **verde claro**
- Interior de Rio Grande do Sul

**Interpretación:**
- Puerto Alegre (costa) funciona bien → azul/verde
- Problema está en interior montañoso (serras gaúchas)
- Hub posicionado para servir esas zonas de difícil acceso

**Validación:**
- ✅ No está en Porto Alegre (redundante con infraestructura existente)
- ✅ Cubre vacío geográfico del interior

---

#### 🏭 **Hub #4: Vitória do Jari, AP (Norte)**

**Observación en el mapa:**
- Está en zona **azul/verde** (aparentemente sin problemas visibles)
- Muy al norte, cerca de Guyana

**Interpretación crítica:**
- Esta es la zona con **menos densidad de datos**
- El heatmap no muestra mucho rojo porque hay **pocas entregas históricas**
- El algoritmo lo detectó porque las pocas órdenes (306) tienen retrasos altos (11.5 días)

**Validación:**
- ⚠️ Cuidado: Baja densidad de datos = predicción menos confiable
- ⚠️ Confirma recomendación de "evaluar antes de ejecutar"

---

### Criterio de Optimización (Cómo Funciona)

**Fórmula que minimiza K-Means:**
```
Minimizar: Σ [(días_retraso)² × distancia²(orden_i, hub_j)]
          i=1 a 7,239

Donde:
- días_retraso² = Peso cuadrático del dolor
- distancia² = Penalización por lejanía
```

**Ejemplo numérico (Hub #1):**
```
Orden en Poços de Caldas (MG):
  - Retraso: 12 días → peso = 144
  - Distancia a Baependi: 80 km
  - Contribución: 144 × 80² = 921,600

Orden en Guarulhos (SP):
  - Retraso: 3 días → peso = 9
  - Distancia a Baependi: 250 km
  - Contribución: 9 × 250² = 562,500

K-Means encuentra coordenadas que minimizan suma de TODAS estas contribuciones
```

**Por qué funciona:**
- Pedidos con retrasos largos "jalan" más fuerte los centroides
- Balance entre cercanía y gravedad del problema
- Resultado: Hubs en posiciones estratégicas, no necesariamente en zonas más rojas

---

### Cobertura y Superposición

**Observaciones clave:**

1. **Los círculos NO se superponen significativamente**
   - ✅ Buena señal: Zonas bien diferenciadas
   - ✅ No hay redundancia entre hubs

2. **Hub #1 tiene el círculo más grande visualmente**
   - Correcto: Cubre 66.3% del problema
   - Zona más densa de incidentes

3. **Hub #4 está muy alejado del resto**
   - Refleja realidad geográfica: Norte está aislado
   - Justifica recomendación de "evaluar viabilidad"

---

### Impacto Esperado

**Con 3 hubs (excluir #4):**

| Métrica | Actual | Proyectado | Mejora |
|---------|--------|------------|--------|
| **Retraso promedio en zonas cubiertas** | 10.5 días | ~5.8 días | -45% |
| **% órdenes atendidas <7 días** | 35% | 68% | +33 pp |
| **Cobertura del problema** | N/A | 95.8% | - |

**Supuestos:**
- Hub reduce distancia promedio ~50%
- Tiempo tránsito proporcional a distancia (validado por GP espacial)
- No considera mejoras en frecuencia de rutas

---

### Limitaciones del Análisis

**1. Radio de 300 km es arbitrario:**
- Basado en estándares logísticos brasileños
- NO calculado por el algoritmo
- En realidad, puede variar: 150-500 km según geografía

**2. K-Means asume K=4:**
- ¿Por qué 4 y no 3 o 5?
- Decisión tomada a priori (no optimizada)
- **Mejora futura:** Usar Elbow Method para elegir K óptimo

**3. No considera costos:**
- Algoritmo solo minimiza distancia ponderada
- No sabe cuánto cuesta un hub en cada zona
- Análisis de viabilidad económica es paso separado

**4. No considera restricciones operativas:**
- Disponibilidad de inmuebles
- Regulaciones locales
- Acceso a transportistas

---

## 🎯 Conclusiones de la Fase

### Lo que el algoritmo SÍ hizo bien:

✅ Identificó zonas geográficas cohesivas con problemas  
✅ Priorizó gravedad (retraso largo) sobre frecuencia  
✅ Encontró posiciones balanceadas (no extremos)  
✅ Cobertura de 95.8% con solo 3 hubs  

### Lo que requiere validación adicional:

⚠️ **Hub #4:** Baja densidad de datos, evaluar económicamente  
⚠️ **Número de hubs:** ¿4 es óptimo o arbitrario?  
⚠️ **Radio 300 km:** Validar con datos reales de rutas  
⚠️ **Costos:** Análisis CAPEX/OPEX pendiente  

### Próximos pasos recomendados:

**Corto plazo (0-3 meses):**
1. ✅ Estudio de factibilidad en Baependi (Hub #1)
2. ✅ Benchmarking de costos logísticos por región
3. ✅ Análisis de sensibilidad: ¿Qué pasa con K=3 o K=5?

**Mediano plazo (3-6 meses):**
1. 🏗️ Piloto en Hub #1 si viable
2. 📊 Monitoreo de KPIs: retraso, costo, volumen
3. 🔄 Recalibrar ubicaciones con datos del piloto

**Largo plazo (6-12 meses):**
1. 🏗️ Expansión a Hub #2 y #3 si piloto exitoso
2. 🔁 Reentrenamiento con datos actualizados
3. 🌐 Explorar Hub #4 si volumen aumenta

---

---
# 🕸️ PARTE 8: ANÁLISIS DE REDES LOGÍSTICAS
---

## Objetivo

Modelar el sistema logístico de Olist como un **grafo dirigido** para identificar:

- 🏭 **Ciudades clave** como productores y consumidores
- 🌉 **Puentes críticos** (ciudades intermediarias esenciales)
- ⚠️ **Vulnerabilidades sistémicas** (dependencias peligrosas)
- 📊 **Concentración de riesgo** (monocentrismo)

**Pregunta de negocio:**
> "¿Qué tan frágil es nuestra red logística? ¿Qué ciudades, si fallan,
> paralizan el sistema?"

---

## Metodología: Teoría de Grafos

### Construcción del Grafo Dirigido

**Elementos del grafo:**
- **Nodos (vértices):** Ciudades únicas (sellers y customers)
- **Aristas dirigidas:** Rutas comerciales (Seller → Customer)
- **Pesos:** Volumen de órdenes por ruta

**Ejemplo visual:**
```
São Paulo ──(2,411)──> Rio de Janeiro
     │
     └──(1,977)──> Belo Horizonte
     
Ibitinga ──(1,977)──> São Paulo
```

**Herramienta:** NetworkX (biblioteca estándar de Python para análisis de redes)

---

### Métricas de Centralidad

Usamos 3 métricas clásicas de teoría de grafos:

#### **1. Out-Degree Centrality (Productores)**

**¿Qué mide?**
- A cuántas ciudades DIFERENTES vende un nodo

**Fórmula:**
```
out_degree(ciudad) = (# ciudades a las que vende) / (total ciudades - 1)
```

**Interpretación:**
- 59.3% = São Paulo vende a 59.3% de TODAS las ciudades destino
- **NO** es el 59.3% del volumen de órdenes
- **SÍ** es el 59.3% de la diversidad geográfica

**Por qué importa:**
- Alta centralidad = Dependencia sistémica
- Si falla, muchas ciudades quedan sin proveedor

---

#### **2. In-Degree Centrality (Consumidores)**

**¿Qué mide?**
- De cuántas ciudades DIFERENTES compra un nodo

**Fórmula:**
```
in_degree(ciudad) = (# ciudades vendedoras) / (total ciudades - 1)
```

**Interpretación:**
- 10.1% = São Paulo compra desde 10.1% de las ciudades vendedoras
- Indica diversificación de proveedores

---

#### **3. Betweenness Centrality (Puentes)**

**¿Qué mide?**
- Cuántos "caminos" entre otras ciudades pasan por este nodo

**Fórmula:**
```
betweenness(nodo) = Σ (caminos más cortos que pasan por nodo) / (total caminos)
```

**Interpretación:**
- 2.30% = Cotia está en 2.30% de los caminos más cortos
- **Ciudad "puente"** - su falla rompe conexiones

**Ejemplo práctico:**
```
Campinas quiere comprar de Marília
   → Camino más corto: Marília → Cotia → Campinas
   → Si Cotia falla, el camino se duplica
```

**Nota técnica:**
- Calculamos betweenness solo en **top 100 ciudades** (por eficiencia computacional)
- Captura >90% del volumen de la red

---

In [ ]:
# Cargar datos adicionales necesarios
print("📦 Cargando datos para análisis de red...")

import pandas as pd

base_path = str(PROJECT_ROOT / 'data' / 'raw')

df_sellers = pd.read_csv(f'{base_path}/olist_sellers_dataset.csv')
df_items = pd.read_csv(f'{base_path}/olist_order_items_dataset.csv')
df_customers = pd.read_csv(f'{base_path}/olist_customers_dataset.csv')

print("✅ Datos cargados")

In [ ]:
# Setup path (si no se ha hecho)
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Forzar recarga
import importlib
if 'src.network.graph_analytics' in sys.modules:
    importlib.reload(sys.modules['src.network.graph_analytics'])

# Construir y analizar red
from src.network.graph_analytics import NetworkAnalyzer

print("🕸️ Construyendo red logística...")

# Inicializar
analyzer = NetworkAnalyzer(
    df_orders=df_main,
    df_sellers=df_sellers,
    df_items=df_items,
    df_customers=df_customers
)

# Construir grafo
graph = analyzer.build_graph()

# Calcular métricas
metrics = analyzer.calculate_centrality(top_n=100)

print("\n✅ Red analizada")

In [ ]:
# Generar reporte de vulnerabilidad
report = analyzer.get_vulnerability_report()

analyzer.print_report(report)

In [ ]:
# Visualizar top rutas
top_routes = analyzer.visualize_top_routes(n=15)

display(top_routes)

---

## 📊 Interpretación de Resultados

### Estadísticas de la Red

**Datos procesados:**
- **Total órdenes:** 109,908
- **Nodos únicos:** ~4,000 ciudades, municipios o pueblos (sellers + customers)
- **Aristas únicas:** ~5,000 rutas comerciales
- **Red tipo:** Dirigida (seller → customer), ponderada (volumen)

---

### 🏭 Análisis de Productores (Out-Degree Centrality)

#### **TOP 5 CIUDADES PRODUCTORAS:**

| Rank | Ciudad | Out-Degree | Interpretación |
|------|--------|-----------|----------------|
| 1 | São Paulo | 59.3% | Vende a 59.3% de todas las ciudades destino |
| 2 | Ibitinga | 30.4% | Vende a 30.4% de ciudades destino |
| 3 | Santo André | 18.7% | Vende a 18.7% de ciudades destino |
| 4 | Curitiba | 17.6% | Vende a 17.6% de ciudades destino |
| 5 | Belo Horizonte | 17.0% | Vende a 17.0% de ciudades destino |

---

#### **¿Qué significa realmente 59.3%?**

**Cálculo técnico:**
```python
out_degree_centrality = (# ciudades a las que vende) / (total_ciudades - 1)

São Paulo:
  Vende a: ~2,372 ciudades diferentes
  Total ciudades: ~4,000
  
  out_degree = 2,372 / 3,999 = 0.593 = 59.3%
```

**Interpretación de negocio:**

✅ **Lo que SÍ significa:**
- São Paulo es el proveedor más **geográficamente diversificado**
- Tiene alcance nacional (llega a casi 6 de cada 10 ciudades)
- Es el nodo con mayor **conectividad saliente**

❌ **Lo que NO significa:**
- NO es el 59.3% del volumen total de órdenes
- NO significa que 59% de las órdenes salen de São Paulo
- Es una medida de **diversidad**, no de **volumen**

---

#### **🚨 Diagnóstico: Monocentrismo Crítico**

**Problema identificado:**
```
São Paulo: 59.3%  ──────┐
Ibitinga: 30.4%   ──────┤ → Brecha de 28.9 puntos
                        │    (casi el DOBLE)
Santo André: 18.7% ─────┘
```

**¿Por qué es crítico?**

1. **Concentración extrema:**
   - São Paulo tiene 2× la conectividad del #2
   - Brecha de ~30 puntos porcentuales
   - Las otras ciudades son mucho más locales

2. **Vulnerabilidad sistémica:**
```
   Si São Paulo falla (huelga, desastre, bloqueo):
     → ~2,372 ciudades quedan sin proveedor principal
     → 59.3% de la red afectada
     → Redistribución masiva necesaria
```

3. **Dependencia en cascada:**
   - Otras ciudades productoras (Ibitinga, Santo André) también dependen de SP
   - Ver top rutas: "Ibitinga → São Paulo: 1,977 órdenes"
   - Si SP cae, los redistribuidores también colapsan

---

#### **Caso especial: Ibitinga (30.4%)**

**¿Qué es Ibitinga?**
- Ciudad pequeña en interior de São Paulo (pop. ~40k)
- **Especialización:** Industria textil (ropa de cama, toallas)

**¿Por qué tan alto out-degree?**
- ✅ Cluster industrial especializado
- ✅ Vende a todo Brasil (producto de nicho con demanda nacional)
- ✅ NO es redistribuidor de São Paulo - es productor directo

**Lección:**
> Conectividad NO implica tamaño de ciudad. Ibitinga es pequeña pero
> altamente conectada por especialización industrial.

---

### 🛒 Análisis de Consumidores (In-Degree Centrality)

#### **TOP 5 CIUDADES CONSUMIDORAS:**

| Rank | Ciudad | In-Degree | Interpretación |
|------|--------|-----------|----------------|
| 1 | São Paulo | 10.1% | Compra desde 10.1% de ciudades vendedoras |
| 2 | Rio de Janeiro | 8.3% | Compra desde 8.3% de ciudades vendedoras |
| 3 | Belo Horizonte | 7.0% | Compra desde 7.0% de ciudades vendedoras |
| 4 | Brasília | 5.9% | Compra desde 5.9% de ciudades vendedoras |
| 5 | Curitiba | 5.4% | Compra desde 5.4% de ciudades vendedoras |

---

#### **Interpretación:**

**São Paulo como consumidor (10.1%):**
```
Compra desde: ~404 ciudades vendedoras diferentes
Total vendedores: ~4,000

in_degree = 404 / 3,999 = 0.101 = 10.1%
```

**¿Por qué es importante?**

✅ **Diversificación de proveedores:**
- São Paulo NO depende de una sola ciudad vendedora
- 404 proveedores diferentes = alta resiliencia
- Si un proveedor falla, hay 403 alternativas

⚠️ **Concentración de demanda:**
- Top 5 consumidoras suman ~37% de diversidad
- Estas 5 ciudades absorben mayoría de flujos entrantes
- Refuerza patrón: Sur/Sudeste domina consumo

---

### 🌉 Análisis de Puentes (Betweenness Centrality)

#### **TOP 5 CIUDADES PUENTE:**

| Rank | Ciudad | Betweenness | Interpretación |
|------|--------|-------------|----------------|
| 1 | Cotia | 2.30% | Intermediaria en 2.30% de caminos |
| 2 | São Gonçalo | 1.84% | Intermediaria en 1.84% de caminos |
| 3 | Fortaleza | 1.77% | Intermediaria en 1.77% de caminos |
| 4 | Foz do Iguaçu | 1.70% | Intermediaria en 1.70% de caminos |
| 5 | Araraquara | 1.56% | Intermediaria en 1.56% de caminos |

---

#### **¿Qué es un "puente" en redes?**

**Definición técnica:**
- Ciudad que está en el **camino más corto** entre otras ciudades
- Su remoción **desconecta** partes de la red o aumenta distancias

**Ejemplo con Cotia (2.30%):**
```
Escenario: Osasco quiere comprar de Barueri

Con Cotia funcionando:
  Osasco → Cotia → Barueri (2 saltos)
  
Sin Cotia:
  Osasco → São Paulo → Guarulhos → Barueri (3+ saltos)
  Distancia aumenta 50%+
```

**Cálculo:**
```
betweenness(Cotia) = (caminos que pasan por Cotia) / (total caminos en subgrafo top 100)
                   = 2.30%
```

---

#### **¿Por qué Cotia es un puente?**

**Geografía estratégica:**
- Cotia está en **región metropolitana de São Paulo**
- A 30 km del centro de SP
- Punto intermedio entre ciudades del interior y capital

**Función logística:**
- Centro de **cross-docking** natural
- Redistribución de carga São Paulo ↔ Interior
- Evita congestión en el centro de SP

**Vulnerabilidad:**
```
Si Cotia falla:
  → 2.30% de rutas necesitan reorganización
  → Aumenta distancia promedio
  → Costos de transporte suben ~15-25%
```

---

#### **Caso especial: Foz do Iguaçu (1.70%)**

**¿Por qué es puente si está en la frontera?**

- Frontera triple: Brasil-Argentina-Paraguay
- **Hub de comercio internacional** para Olist
- Punto de entrada/salida para mercancías importadas

**Función:**
```
Productos importados (China, etc.):
  Paraguay → Foz do Iguaçu → Brasil interior
  
Si Foz falla:
  → Redirigir a Santos (SP) - puerto más caro
  → +3-5 días de tránsito
```

---

### 🛣️ Análisis de Supercarreteras (Top Rutas)

#### **TOP 15 RUTAS POR VOLUMEN:**

**Patrón dominante: Gravitación hacia São Paulo**

| Origen | Destino | Volumen | Patrón |
|--------|---------|---------|--------|
| São Paulo | São Paulo | 6,942 | 🔄 Auto-consumo |
| São Paulo | Rio de Janeiro | 2,411 | 📤 Distribución nacional |
| Ibitinga | São Paulo | 1,977 | 📥 Consolidación textil |
| São Paulo | Belo Horizonte | 1,088 | 📤 Distribución nacional |
| Ibitinga | Rio de Janeiro | 898 | 📤 Distribución directa |

---

#### **Hallazgo #1: São Paulo → São Paulo (6,942 órdenes)**

**¿Qué significa?**

- **NO** es error de datos
- Vendedores en São Paulo vendiendo a consumidores en São Paulo
- **Lógica:** Ciudad metropolitana de 12M+ habitantes

**Implicación logística:**
```
Last-mile dentro de SP:
  - Distancia: 5-50 km
  - Tiempo entrega: 1-2 días
  - Costo bajo

Estrategia:
  ✅ Micro-hubs en zonas de SP (Zona Sul, Norte, Leste)
  ✅ Same-day delivery viable
```

---

#### **Hallazgo #2: Ibitinga como segundo origen (1,977 + 898 órdenes)**

**Validación del análisis de productores:**
```
Ibitinga → São Paulo: 1,977 órdenes
Ibitinga → Rio: 898 órdenes
Total: 2,875 órdenes

¿Por qué tan alto volumen?
  → Especialización textil = demanda constante
  → Cluster industrial compacto = economías de escala
```

---

#### **Hallazgo #3: Rutas radiales desde São Paulo**

**Patrón identificado:**
```
São Paulo como epicentro:

SP → Rio (2,411)         ─┐
SP → BH (1,088)          ─┤
SP → Goiânia (642)       ─┤ → Distribución nacional desde SP
SP → Brasília (615)      ─┤
SP → Curitiba (602)      ─┤
SP → Porto Alegre (573)  ─┘
```

**Interpretación:**
- São Paulo actúa como **mega-hub de redistribución**
- Flujo: Interior → SP → Capitales regionales
- Modelo hub-and-spoke clásico

---

## ⚠️ Reporte de Vulnerabilidad

### Riesgo #1: Dependencia São Paulo (CRÍTICO)

**Nivel de riesgo:** 🔴 **CRÍTICO**

**Indicadores:**
```
Out-degree: 59.3%  (distribuye a 6 de cada 10 ciudades)
In-degree: 10.1%   (concentra demanda)
Rutas top: 7 de 15 involucran SP
```

**Escenarios de falla:**

| Evento | Probabilidad | Impacto | Mitigación |
|--------|--------------|---------|------------|
| Huelga transportistas | Media (5%/año) | Alto | Hubs satélite (Campinas, Santos) |
| Bloqueo carreteras | Media (10%/año) | Medio | Rutas alternativas pre-mapeadas |
| Desastre natural | Baja (<1%/año) | Crítico | Redundancia geográfica (Curitiba, BH) |

---

### Riesgo #2: Puentes sin Redundancia (MEDIO)

**Nivel de riesgo:** 🟡 **MEDIO**

**Ciudades puente críticas:**
- Cotia (2.30%)
- Foz do Iguaçu (1.70%)

**Problema:**
- Si fallan, NO hay alternativas directas
- Reorganización de rutas costosa (+ 20-40% costo)

**Recomendación:**
```
Crear centros de redistribución alternativos:
  - Cotia backup → Barueri o Osasco
  - Foz backup → Cascavel (PR)
```

---

### Riesgo #3: Monocentrismo Industrial (MEDIO-BAJO)

**Nivel de riesgo:** 🟡 **MEDIO-BAJO**

**Concentración textil en Ibitinga:**
- 30.4% out-degree
- Producto específico (ropa de cama)

**¿Qué pasa si Ibitinga falla?**
- Productos textiles escasean
- Alternativa: Importar de otros estados (SC, CE)
- Impacto: +15-30% costo en categoría específica

**Mitigación:**
- Diversificar proveedores textiles
- Mapear clusters alternativos (Fortaleza, Blumenau)

---

## 💡 Conclusiones y Recomendaciones

### Hallazgos Clave

1. ✅ **Red tipo hub-and-spoke dominante**
   - São Paulo es mega-hub central
   - Modelo eficiente pero vulnerable

2. ✅ **Especialización geográfica funciona**
   - Ibitinga (textil) es caso exitoso
   - Fomenta diversificación por nicho

3. ⚠️ **Vulnerabilidad sistémica alta**
   - 59.3% de red depende de un nodo
   - Necesidad urgente de redundancia

---

### Recomendaciones Estratégicas

#### **Corto Plazo (0-6 meses):**

1. **Mapeo de rutas alternativas:**
   - Si SP falla, ¿cómo redistribuir desde Curitiba/BH?
   - Documentar protocolos de contingencia

2. **Monitoreo de puentes críticos:**
   - Dashboard en tiempo real: Cotia, Foz do Iguaçu
   - Alertas si volumen cae >30%

3. **Contratos de redundancia:**
   - Proveedores backup en categorías críticas
   - Pre-negociar capacidad en Curitiba, Campinas

---

#### **Mediano Plazo (6-12 meses):**

1. **Descentralización gradual:**
   - Abrir micro-hubs en Belo Horizonte, Curitiba
   - Objetivo: Reducir dependencia SP de 59% → 40%

2. **Diversificación de proveedores:**
   - Onboarding de sellers en Nordeste
   - Reducir concentración Sur/Sudeste

3. **Análisis predictivo:**
   - Simular fallos de nodos críticos
   - Quantificar impacto financiero

---

#### **Largo Plazo (1-2 años):**

1. **Red mallada (mesh network):**
   - Pasar de hub-and-spoke a múltiples hubs
   - Conexiones directas entre ciudades secundarias

2. **Redundancia geográfica:**
   - Hubs estratégicos en cada región
   - Norte: Manaus, Nordeste: Fortaleza, etc.

3. **Optimización dinámica:**
   - Machine Learning para reasignación automática
   - Si nodo falla, redistribución en <2 horas

---

---
# 📊 ANEXO: COMPARACIÓN CON ESTÁNDARES DE LA INDUSTRIA
---

## Contexto

Este proyecto implementa metodologías de **optimización logística** y **análisis de redes**.
Para validar su relevancia profesional, comparamos nuestro enfoque con los estándares actuales
utilizados por empresas líderes del sector.

---

## 🏗️ PARTE 1: Facility Location (Optimización de Hubs)

### Métodos Estándar en la Industria

#### **1. Center of Gravity (Método Clásico)**

**Empresas que lo usan:** Amazon (sistemas legacy años 2000s), Walmart

**Algoritmo:**
```
X_hub = Σ(volumen_i × latitud_i) / Σ(volumen_i)
Y_hub = Σ(volumen_i × longitud_i) / Σ(volumen_i)
```

**Concepto:** Centro de masa ponderado por volumen de ventas

**✅ Ventajas:**
- Matemática simple (Excel es suficiente)
- Rápido (<1 segundo)
- Interpretable (centro de masa literal)

**❌ Desventajas:**
- Pondera por VOLUMEN, no por PROBLEMA
- Asume costos simétricos (irreal en logística)
- No considera restricciones (ríos, zonas prohibidas, topografía)

**Uso actual:** Análisis preliminar, no para decisión final

---

#### **2. P-Median Problem (Optimización Matemática)**

**Empresas que lo usan:** UPS, FedEx, DHL

**Formulación:**
```
Minimizar: Σ Σ distancia_ij × asignación_ij
           i j

Sujeto a:
  - Σ y_j = p  (exactamente p hubs)
  - x_ij ≤ y_j (asignar solo a hubs abiertos)
  - x_ij ∈ {0,1}
```

**Herramientas:** CPLEX, Gurobi (programación entera mixta)

**✅ Ventajas:**
- Solución óptima **garantizada** matemáticamente
- Considera restricciones complejas (capacidad, presupuesto)
- Múltiples objetivos simultáneos (costo + tiempo)

**❌ Desventajas:**
- Computacionalmente **NP-hard** (exponencial)
- Requiere licencias comerciales ($10k-50k USD/año)
- Difícil de explicar a gerencia no técnica ("caja negra")
- Tiempo de cómputo: Horas para >1,000 puntos

**Uso actual:** Empresas grandes con presupuestos IT altos

---

#### **3. Capacitated Facility Location**

**Empresas que lo usan:** Mercado Libre, Alibaba

**Extensión del P-Median:**
```
Añade restricción:
  Σ demanda_i × x_ij ≤ capacidad_j × y_j
  i
```

**Considera:**
- Tamaño del almacén (m²)
- Personal disponible
- Horas operativas
- Throughput máximo

**✅ Ventajas:**
- Más realista (hubs tienen límites físicos)
- Previene sobrecarga

**❌ Desventajas:**
- Aún más complejo que P-Median
- Requiere datos precisos de capacidad (difícil de obtener)

---

#### **4. Weighted K-Means (Machine Learning) ← NUESTRO MÉTODO**

**Empresas que lo usan:** Startups, proyectos de investigación

**Algoritmo:**
```python
weights = delay_days ** 2.0
kmeans.fit(coordenadas, sample_weight=weights)
```

**Concepto:** Clustering ponderado por gravedad del problema

**✅ Ventajas:**
- **Gratis** (sklearn open-source)
- **Rápido** (1-5 segundos para 7k puntos)
- **Flexible** (cambiar función de peso es trivial)
- **Interpretable** (clusters visualizables en mapa)
- **Escalable** (funciona con millones de puntos)

**❌ Desventajas:**
- **NO garantiza óptimo global** (heurística)
- Sensible a inicialización (mitigado con `n_init=10`)
- No considera restricciones geográficas nativas

**Uso actual:** Prototipado rápido, análisis exploratorio, MVPs

---

### 📊 Comparación: Nuestro Método vs Industria

| Aspecto | Industria (P-Median) | Nuestro (K-Means) | Veredicto |
|---------|---------------------|-------------------|-----------|
| **Optimalidad** | ✅ Garantizada | ⚠️ Heurística | Gap moderado |
| **Costo computacional** | ❌ Alto (horas) | ✅ Bajo (segundos) | **Ganamos** |
| **Costo de software** | ❌ $10k-50k/año | ✅ Gratis | **Ganamos** |
| **Explicabilidad** | ❌ Caja negra | ✅ Visual/intuitivo | **Ganamos** |
| **Restricciones reales** | ✅ Capacidad, costos | ❌ No nativas | Gap grande |
| **Escalabilidad** | ⚠️ <10k puntos | ✅ >100k puntos | **Ganamos** |

---

### 💡 Nuestra Posición Estratégica

**Categoría:** Análisis exploratorio / MVP

**¿Cuándo es apropiado nuestro método?**

✅ **SÍ usar K-Means ponderado cuando:**
- Proyecto en fase de validación (no producción crítica)
- Presupuesto limitado (<$50k para software)
- Necesidad de resultados rápidos (horas, no semanas)
- Audiencia no técnica (requiere explicación visual)
- Datos históricos incompletos (restricciones desconocidas)

❌ **NO usar K-Means cuando:**
- Decisión de inversión >$1M (requiere óptimo garantizado)
- Restricciones críticas (capacidad, regulaciones)
- Red de >100 hubs (complejidad combinatoria)
- Empresa con presupuesto para Gurobi/CPLEX

---

### 🎯 Cómo Escalar a Nivel Industria

**Path de migración:**
```
Fase 1 (ACTUAL): K-Means exploratorio
  ↓ Validar viabilidad de negocio
  
Fase 2: Añadir restricciones simples
  ↓ Filtrar zonas prohibidas manualmente
  ↓ Validar con gerentes regionales
  
Fase 3: Implementar P-Median
  ↓ Usar resultados K-Means como "warm start"
  ↓ Refinar con solver profesional
  
Fase 4: Producción
  ↓ Sistema híbrido: K-Means (exploración) + P-Median (decisión)
```

---

## 🕸️ PARTE 2: Network Analytics (Análisis de Grafos)

### Métodos Estándar en la Industria

#### **1. Degree Centrality ← USAMOS ESTE**

**Empresas:** Amazon, UPS, Walmart, FedEx (TODAS)

**Fórmula:**
```
out_degree(v) = (# aristas salientes) / (total_nodos - 1)
```

**Historia:** Método desde 1960s, validado por 60+ años

**✅ Estado:**
- ✅ **Estándar de oro** de la industria
- ✅ Complejidad O(n) - muy eficiente
- ✅ Interpretación directa

**Veredicto:** Nuestro uso es **100% correcto e industrial**

---

#### **2. Betweenness Centrality ← USAMOS ESTE (optimizado)**

**Empresas:** FedEx, DHL (análisis de hubs críticos)

**Fórmula:**
```
betweenness(v) = Σ (caminos_cortos_pasando_por_v) / (total_caminos)
```

**Complejidad:** O(n³) - **muy costoso**

**Nuestra optimización:**
```python
# Solo calculamos en top 100 nodos (subgrafo)
top_nodes = sorted(graph.degree)[:100]
subgraph = graph.subgraph(top_nodes)
betweenness = nx.betweenness_centrality(subgraph)
```

**Resultado:**
- Reduce de O(4000³) a O(100³)
- Tiempo: 5 segundos vs 2 horas
- Captura >90% del volumen de red

**✅ Veredicto:** Optimización **inteligente** aceptada en industria

---

#### **3. PageRank (Algoritmo de Google)**

**Empresas:** Google, Alibaba, eBay

**Fórmula:**
```
PR(v) = (1-d)/n + d × Σ PR(u)/out_degree(u)
                      u→v
```

**Ventaja sobre Degree:**
- Considera **CALIDAD** de conexiones
- Conexión desde nodo importante vale MÁS

**Aplicación logística:**
```
Vender desde São Paulo → Buenos Aires (PR alto)
  vale MÁS que
Vender desde pueblo pequeño → Buenos Aires (PR bajo)
```

**❌ NO LO USAMOS** (pero podríamos agregarlo fácilmente)

**Gap:** Moderado

---

#### **4. Community Detection (Louvain, Label Propagation)**

**Empresas:** Mercado Libre, eBay

**Objetivo:** Encontrar "clusters" naturales en la red

**Aplicación logística:**
```
Detectar regiones comerciales naturales:
  • "Cluster textil Ibitinga-São Paulo-Interior"
  • "Cluster tecnología São Paulo-Campinas"
  • "Cluster agro Sul (RS-SC-PR)"
```

**Beneficio:**
- Optimizar decisiones por región
- Identificar ecosistemas comerciales

**❌ NO LO USAMOS** (oportunidad de mejora)

**Gap:** Moderado

---

#### **5. Network Flow Optimization (Avanzado)**

**Empresas:** UPS (ORION system $250M), Amazon (anticipatory shipping)

**Problemas que resuelve:**
- Max Flow / Min Cut
- Minimum Cost Flow
- Multi-commodity Flow

**Formulación:**
```
Minimizar: Σ costo_ij × flujo_ij
Sujeto a:
  - Conservación de flujo en cada nodo
  - Capacidades de aristas
  - Demandas de origen/destino
```

**Herramientas:** Google OR-Tools, Gurobi

**Nivel:** PhD Operations Research

**❌ NO LO USAMOS** (requiere equipo especializado)

**Gap:** Grande (pero normal para proyectos no-enterprise)

---

### 📊 Comparación: Nuestro Método vs Industria

| Aspecto | Industria (Big Tech) | Nuestro Proyecto | Veredicto |
|---------|---------------------|------------------|-----------|
| **Métricas básicas** | Degree/Betweenness | Degree/Betweenness | ✅ **Igual** |
| **Métricas avanzadas** | PageRank, Eigenvector | No | ⚠️ Gap moderado |
| **Community detection** | Louvain | No | ⚠️ Gap moderado |
| **Flow optimization** | Min-Cost Flow | No | ❌ Gap grande |
| **Análisis temporal** | Streaming (tiempo real) | Batch (histórico) | ❌ Gap grande |
| **Escalabilidad** | >1M nodos (GraphX, Neo4j) | <100k nodos (NetworkX) | ⚠️ Gap moderado |
| **Visualización** | Gephi, Cytoscape | Matplotlib | ⚠️ Gap moderado |

---

### 💡 Nuestra Posición Estratégica

**Categoría:** Análisis descriptivo estándar

**¿Cuándo es apropiado nuestro método?**

✅ **SÍ usar Degree/Betweenness cuando:**
- Análisis exploratorio de red
- Identificar nodos críticos (estándar universal)
- Reportes ejecutivos (métricas comprensibles)
- Redes de <100k nodos
- Análisis histórico (no tiempo real)

❌ **NO suficiente cuando:**
- Optimización de rutas en tiempo real (requiere flow)
- Predicción de cascadas de fallo (requiere simulación)
- Segmentación de clientes (requiere communities)
- Análisis de influencia (requiere PageRank)

---

### 🎯 Cómo Escalar a Nivel Industria

**Mejoras incrementales:**

**Corto plazo (0-3 meses):**
```python
# Añadir PageRank (5 líneas de código)
pagerank = nx.pagerank(graph, weight='weight')

# Detectar communities (10 líneas)
import community as community_louvain
communities = community_louvain.best_partition(graph)
```

**Mediano plazo (3-6 meses):**
- Visualización avanzada (Gephi export)
- Análisis temporal (evolución de red mes a mes)
- Métricas de robustez (simulación de fallos)

**Largo plazo (6-12 meses):**
- Migrar a Neo4j (grafo database)
- Implementar flow optimization (Google OR-Tools)
- Pipeline en tiempo real (Kafka + NetworkX streaming)

---

## 🎓 Conclusión General

### Fortalezas del Proyecto

| Aspecto | Evaluación | Justificación |
|---------|-----------|---------------|
| **Fundamento teórico** | ✅ **Sólido** | Usamos métodos estándar validados |
| **Implementación técnica** | ✅ **Correcta** | NetworkX + sklearn son herramientas profesionales |
| **Escalabilidad inicial** | ✅ **Adecuada** | Maneja datasets medianos (<100k) |
| **Costo-beneficio** | ✅ **Excelente** | 100% open-source, resultados útiles |
| **Explicabilidad** | ✅ **Superior** | Más interpretable que solvers comerciales |

---

### Gaps Identificados (Normales para Proyectos No-Enterprise)

| Gap | Severidad | ¿Crítico? | Mitigación |
|-----|-----------|-----------|------------|
| No garantiza óptimo (K-Means) | ⚠️ Moderada | No | Suficiente para análisis exploratorio |
| Sin restricciones de capacidad | ⚠️ Moderada | No | Validar manualmente con gerentes |
| Sin PageRank/Communities | 🟡 Baja | No | Fácil de añadir (mejora futura) |
| Sin flow optimization | 🟢 Baja | No | Solo necesario en operación 24/7 |
| Sin análisis tiempo real | 🟢 Baja | No | Batch es estándar en análisis histórico |

---

### Posicionamiento Profesional

**Nivel del proyecto:**
```
Junior/Mid-Level Data Scientist ✅
Senior con supervisión ⚠️
Principal/Staff ❌ (requiere optimización garantizada)
```

**Comparable con:**
- Proyectos de consultoría boutique
- MVPs de startups logistics-tech
- Tesis de maestría en OR/Analytics
- Análisis exploratorio en empresas Fortune 500

**NO comparable con:**
- Sistemas de producción de Amazon/UPS
- Soluciones enterprise ($1M+ inversión)
- Software crítico de misión (vidas humanas)

---

### 🚀 Mensaje Final
**"¿Por qué no usé P-Median / PageRank / Flow Optimization?"**



> Empecé con la solución más simple que funciona.
> K-Means ponderado y Degree Centrality son métodos validados por décadas, gratuitos,
> y suficientemente precisos para análisis exploratorio.
>
> Si este proyecto pasara a producción, el siguiente paso sería:
> 1. Usar estos resultados como 'warm start' para un solver P-Median
> 2. Añadir PageRank para priorizar proveedores
> 3. Implementar flow optimization para routing en tiempo real
>
> Pero para validar la viabilidad del negocio, esta aproximación es **correcta y profesional**."

---

---
# 🐳 ANEXO: Generar Modelo Compatible con Docker (OPCIONAL)

> **Nota:** Esta sección es **opcional** y solo necesaria si deseas desplegar la API con Docker.

---

## ¿Por qué esta sección?

El modelo espacial entrenado en este notebook usa la versión de NumPy de Colab (2.x). Sin embargo, Docker usa NumPy 1.26.4 para compatibilidad.

Esta sección re-entrena el modelo con un **subset de 10,000 puntos** (en lugar de 109k) para:
- ✅ Compatibilidad con Docker
- ✅ Menor uso de RAM
- ✅ Entrenamiento más rápido (10 minutos vs a 20 minutos)

**El modelo resultante es funcionalmente equivalente** para propósitos de demostración.

---

## 📋 Instrucciones

1. **Ejecuta TODAS las celdas anteriores primero** (para generar `df_main.parquet`)
2. **Ejecuta las celdas de esta sección**
3. **Descarga el archivo generado:**
   - `models/spatial/latest/gp_spatial.pkl` (760 MB)
4. **Colócalo en tu proyecto local**
5. **Ejecuta Docker:** `docker-compose up`

---


In [ ]:
# ==============================================================================
# ⚠️ AVISO IMPORTANTE: ESTA CELDA TOMA ~30-60 SEGUNDOS EN EJECUTARSE
# El archivo de modelo generado ocupará ~700 MB de espacio
# ==============================================================================

# ============================================================================
# RE-ENTRENAR MODELO ESPACIAL (OPCIONAL)
# ============================================================================

# print("="*80)
# print("🔧 ENTRENAMIENTO MODELO ESPACIAL")
# print("="*80)

# Usar PROJECT_ROOT del notebook
# project_root = PROJECT_ROOT
# print(f"📁 Proyecto: {project_root}")

# --- Verificar datos ---
# print("\n📊 Verificando datos...")

# if not (project_root / 'checkpoints' / 'df_main.parquet').exists():
#     print("❌ No se encontró df_main.parquet")
#     print("💡 Primero ejecuta las celdas anteriores para generar los datos")
# else:
#     df_main = pd.read_parquet(project_root / 'checkpoints' / 'df_main.parquet')
#     print(f"   ✅ df_main cargado: {df_main.shape}")

    # --- Crear subset estratégico ---
#     print("\n✂️ Creando subset (10k puntos)...")

#     df_spatial = df_main[df_main['delay_days'].notna()].copy()

#     np.random.seed(42)
#     sample_indices = np.random.choice(
#         len(df_spatial),
#         size=min(10000, len(df_spatial)),
#         replace=False
#     )

#     df_subset = df_spatial.iloc[sample_indices].copy()

#     X_train = df_subset[['geolocation_lat', 'geolocation_lng']].values
#     y_train = df_subset['delay_days'].values

#     print(f"   ✅ Subset: {len(X_train):,} puntos")
#     print(f"   📊 Delay promedio: {y_train.mean():.2f} días")

    # --- Entrenar modelo ---
#     print("\n🔧 Entrenando modelo GP...")
#     print("   (Toma ~30-60 segundos)")

#     from sklearn.gaussian_process import GaussianProcessRegressor
#     from sklearn.gaussian_process.kernels import RBF, ConstantKernel

#     kernel = ConstantKernel(1.0) * RBF(
#         length_scale=329.0,
#         length_scale_bounds=(100, 1000)
#     )

#     gp_spatial = GaussianProcessRegressor(
#         kernel=kernel,
#         alpha=2.0,
#         n_restarts_optimizer=2,
#         normalize_y=True,
#         random_state=42
#     )

#     gp_spatial.fit(X_train, y_train)

#     print(f"   ✅ Modelo entrenado")

    # --- Guardar modelo ---
#     print("\n💾 Guardando modelo...")

#     models_dir = project_root / 'models' / 'spatial' / 'latest'
#     models_dir.mkdir(parents=True, exist_ok=True)

#     model_path = models_dir / 'gp_spatial.pkl'
#     joblib.dump(gp_spatial, model_path)

#     print(f"   ✅ Guardado en: {model_path}")

#     print("\n" + "="*80)
#     print("✅ MODELO ESPACIAL GENERADO")
#     print("="*80)


---

## ✅ Verificación

Si todo funcionó correctamente, deberías ver:
```
✅ MODELO COMPATIBLE CON DOCKER GENERADO

🐳 SIGUIENTE PASO - USAR CON DOCKER
```

El modelo generado (`gp_spatial.pkl` ~760 MB) está optimizado para:
- ✅ NumPy 1.26.4 (compatible con Docker)
- ✅ RAM reducida (10k puntos vs 109k)
- ✅ Predicciones funcionales

**Nota:** Este modelo tiene precisión ligeramente menor que el completo, pero es suficiente para demostración de la API.

---
